# TARGET-DDI — full from-scratch reproduction (paper code artifact)

Reproduces the results in *Overcoming the Cold-Start Barrier: From Molecular Fingerprints to
Protein-Target Knowledge in TARGET-DDI*.

**Run from scratch:** Kaggle/Colab with **GPU + Internet ON**, then Run All. Do NOT attach any
checkpoint dataset -- with an empty working directory the pipeline downloads DrugBank-DDI (TDC),
fetches UniProt/GO/Pfam annotations, computes ChemBERTa + ESM-2 embeddings and all features, then
trains every configuration across 3 seeds on S1/S2/S3, runs paired-bootstrap CIs, and writes
`all_results.csv`. Expect several hours end to end.

**Outputs persist:** the final cell copies all results into `/kaggle/working/ddi_out/`, which Kaggle
keeps on commit (the internal workdir `~/ddi_work` is not persisted). After the run finishes,
**Save Version** to keep `all_results.csv` and the per-unit files.

**Expected headline (3-seed mean micro-AP):** S1 ~0.99 all configs (target-vs-no-target +0.0018);
S2 no-target 0.680 -> full-target 0.762 (bootstrap CI [+0.114,+0.120]); S3 0.396 -> 0.520
(bootstrap CI [+0.148,+0.167]).

This is a faithful reproduction of the published pipeline. The protein frequency filter is computed
over all drugs (as in the paper); see the paper's limitations for the leakage discussion.

# RESUME — run only what's missing (TARGET-DDI)

This is **not a rewrite**. Cells 1–14 below are your original notebook's setup
(config, environment, target-data fetch, splits/metrics, molecular features, model
classes, self-test) reused verbatim — a fresh Kaggle session needs them re-executed to
rebuild everything in RAM, even though your checkpoints/results already exist on disk.
Nothing in that block re-does finished work: `run_unit` already skips any unit with a
completed `res_*.json`, and `train_one` already resumes mid-training from `ck_*.pt`
epoch-by-epoch. That logic was already in your notebook — this file just calls it in
the right order.

**What's different from your original notebook:**
1. The run cell (old "Section 8") is replaced with a **priority-ordered, missing-only**
   loop: S3 core ablation ladder first (this is where your paper's central claim has to
   hold), then S3 optional variants, then S1 core, then S1 optional — instead of one
   flat loop that re-checks everything in the original variant order. A wall-clock
   budget guard stops the loop *between* units (never mid-training) so a session that
   runs out of time leaves the checkpoint of whatever was in progress in a safe,
   resumable state.
2. `graph_target_sider` and `dex_ddi` are **not included** — they aren't in the
   notebook you uploaded to me, only in your run logs. Section 3b below is a clearly
   marked slot to paste that variant's setup code; the priority loop picks it up
   automatically once it's there, with no other changes needed.
3. Section 10 (mechanism attribution) is **fixed, not just resumed**: the original
   loads a `ck_..._s{seed}.pt.best` file that is never written anywhere — `train_one`
   only ever writes `ck_{tag}.pt`, and the best weights live inside it at
   `checkpoint["best_state"]["model"]`. Fixed below.
4. A new Section 11 zips the results/checkpoints for download or a Kaggle-dataset
   commit, so a session that runs out of GPU time doesn't lose progress if the working
   directory itself isn't persistent storage.


# TARGET-DDI — Target-Augmented Cold-Start Drug-Drug Interaction Prediction

The thesis, stated plainly and tested by ablation: molecular features (ChemBERTa +
Morgan) hit a ceiling on cold-start DDI because an unseen drug shares no *molecular
identity* with training drugs — but it often shares **protein targets and enzymes**.
Those shared targets are signal that fingerprints do not contain. This notebook adds a
drug->target/enzyme branch and measures, by clean on/off ablation on the hardest splits
(S2/S3), whether it moves the number.

**Why this and not another architecture:** on S1, six architectures (MLP, GraphSAGE,
gated dual-view, and ablations) all converged to micro-AP ~0.29. When the model does not
matter, the ceiling is set by the *inputs*, not the network. So the lever is a new input,
not a new message-passing scheme. This is that experiment.

**Honest guardrails baked in:**
- A **coverage probe runs first** and HARD-STOPS if too few drugs have target annotations
  (a sparse feature cannot help — better to learn that in 5 minutes than after a day).
- The target branch is a clean ablation (`add_target` on/off); its contribution is
  measured on S2/S3, never asserted.
- S2/S3 are run FIRST — that is where the claim lives and where prior runs never reached.
- Leakage audits, tie-correct metrics, paired-bootstrap CIs, resumable checkpoints — all
  carried over and verified.

**Scope this round:** DrugBank-DDI (86-class). Its drug IDs are DrugBank IDs, which join
exactly to DrugBank's protein/target export — so the target feature is clean and the
cold-start test is honest. TWOSIDES needs a CID->DrugBank bridge and is deferred to a
second phase *only if targets prove out here*.

**Feature dimensions (this revision):** the target branch is no longer one merged blob --
it's split into **target / enzyme / transporter / carrier** channels (pharmacodynamic vs
pharmacokinetic mechanisms are different signals), plus **GO / Pfam** functional annotations
of those proteins (generalizes across proteins the model never saw an exact ID for -- the
direct fix for cold-start on rare proteins). The molecular branch gains **ATC classification**
(full code + level-1 anatomical group) and six cheap **RDKit physchem descriptors**. A
**CYP substrate/inhibitor/inducer** hook is included but inert by default (`CFG.enable_cyp_roles`)
since that specific role data sits behind DrugBank's licensed export, not the free one --
point `CFG.cyp_role_csv` at it if you have institutional access.

With `CFG.run_feature_ablation_ladder=True` you get a 3-point ladder per split instead of a
single on/off: `mol_only_classic` (the original chemberta+morgan baseline) -> `ablation_no_target`
(+ ATC/physchem, still no protein info) -> `full_target` (+ everything). That isolates what the
new drug-intrinsic features (ATC/physchem) buy you versus what protein-target information adds
on top.

**Kaggle:** GPU + Internet ON. Run the probe cell first; it is your green light.

## 0. Kaggle setup — load previous session's checkpoint dataset

You're running directly in Kaggle with the previous session's output attached as an
input dataset (via **+ Add Input** in the notebook editor), so there's no need for
`kaggle.json` credentials or an API download — the files are already mounted read-only
under `/kaggle/input/...`. This cell finds that folder and copies its contents into a
writable working directory (`CFG.workdir` below), since `train_one`/`run_unit` need to
write updated checkpoints and results back out as the run resumes.

**Before running:** attach your checkpoint dataset via **+ Add Input** (top right of the
Kaggle notebook editor) if you haven't already. If its input folder name isn't
`ddi-checkpoint`, edit `DATASET_SLUG` below to match — otherwise the cell will try to
auto-detect it.

In [1]:
# =====================================================================
# CHECKPOINT LOADER (optional) — from-scratch safe.
# From scratch: attach NO dataset; this cell just creates the working dir.
# To resume: attach your checkpoint dataset via '+ Add Input'; it is copied into WORKDIR.
# =====================================================================
import os, shutil
from pathlib import Path

WORKDIR = os.path.expanduser("~/ddi_work")   # matches CFG.workdir in the Config cell
os.makedirs(WORKDIR, exist_ok=True)

INPUT_ROOT = Path("/kaggle/input")
DATASET_SLUG = "ddi-checkpoint"   # edit if your attached dataset folder differs

src_dir = None
if INPUT_ROOT.exists():
    cand = INPUT_ROOT / DATASET_SLUG
    if cand.exists():
        src_dir = cand
    else:
        hits = [p for p in INPUT_ROOT.glob("*") if p.is_dir()
                and (list(p.rglob("ck_*.pt")) or list(p.rglob("res_*.json")))]
        src_dir = hits[0] if hits else None

if src_dir is None:
    print(f"[from-scratch] no checkpoint dataset found under {INPUT_ROOT} -> running from scratch. WORKDIR={WORKDIR}")
else:
    copy_root = src_dir
    nested = src_dir / "ddi_work"
    if nested.exists() and any(nested.iterdir()):
        copy_root = nested
    n=0
    for f in copy_root.rglob("*"):
        if f.is_file():
            dest = Path(WORKDIR) / f.relative_to(copy_root)
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists():
                shutil.copy2(f, dest); n+=1
    print(f"[resume] copied {n} files from {src_dir} into {WORKDIR}")

_nck=len(list(Path(WORKDIR).glob("ck_*.pt"))); _nres=len(list(Path(WORKDIR).glob("res_*.json")))
print(f"WORKDIR has {_nck} checkpoints, {_nres} result files")

Using input dataset folder: /kaggle/input/notebooks/nadiashaarawy/notebook03ad799ee2
Copied 88 new file(s) into /root/ddi_work

--- Files in workdir ---
  __notebook__.ipynb
  __output__.json
  atc.json
  ck_DrugBank_v2multi_S2_ablation_no_target_s0.pt
  ck_DrugBank_v2multi_S2_ablation_no_target_s1.pt
  ck_DrugBank_v2multi_S2_ablation_no_target_s2.pt
  ck_DrugBank_v2multi_S2_dex_ddi_s0.pt
  ck_DrugBank_v2multi_S2_dex_ddi_s1.pt
  ck_DrugBank_v2multi_S2_dex_ddi_s2.pt
  ck_DrugBank_v2multi_S2_full_target_s0.pt
  ck_DrugBank_v2multi_S2_full_target_s1.pt
  ck_DrugBank_v2multi_S2_full_target_s2.pt
  ck_DrugBank_v2multi_S2_graph_both_s0.pt
  ck_DrugBank_v2multi_S2_graph_esm_s0.pt
  ck_DrugBank_v2multi_S2_graph_target_sider_s0.pt
  ck_DrugBank_v2multi_S2_mol_only_classic_s0.pt
  ck_DrugBank_v2multi_S2_shared_target_graph_s0.pt
  ck_DrugBank_v2multi_S2_shared_target_graph_s1.pt
  ck_DrugBank_v2multi_S2_shared_target_graph_s2.pt
  ck_DrugBank_v2multi_S3_full_target_s0.pt
  ck_DrugBank_v2multi_S3

### Fix applied: purge poisoned caches carried in by the resume step

The cell above copies your previous session's checkpoint dataset into `WORKDIR` with `if not dest.exists(): copy`. That previous session's dataset was committed while `ec_go_class.json` / `drug_targets_by_cat.json` held a degenerate, rate-limited result (enzyme=transporter=carrier=0%) -- and because the input dataset is read-only, every new session re-copies that same poisoned result in before the target-fetch cell ever runs. Deleting these files from `WORKDIR` by hand doesn't survive the next session, since this copy step just restores them again. Fixed by removing exactly these three re-fetchable API caches right after the copy, every session, before anything reads them -- they are cheap to regenerate and are the only files responsible for the stuck-at-0% pattern. Checkpoints (`ck_*.pt`) and results (`res_*.json`) are untouched.

In [2]:
# Purge cache files carried in from the attached (read-only) checkpoint dataset that are
# known to go stale from rate-limited runs -- these are cheap to refetch, unlike checkpoints.
import os
_POISONABLE_CACHES = ('drug_targets_by_cat.json', 'ec_go_class.json', 'drug_targets.json')
for _f in _POISONABLE_CACHES:
    _p = os.path.join(WORKDIR, _f)
    if os.path.exists(_p):
        os.remove(_p)
        print('removed stale cache carried in from resume dataset:', _p)
print('cache purge done -- enzyme/transporter/carrier will be refetched fresh this session')


removed stale cache carried in from resume dataset: /root/ddi_work/drug_targets_by_cat.json
cache purge done -- enzyme/transporter/carrier will be refetched fresh this session


## 1. Config

In [3]:
from types import SimpleNamespace
import os
CFG=SimpleNamespace(
    dataset="DrugBank",                 # clean target join this round
    splits=["S2","S3","S1"],            # S2/S3 FIRST (where the claim lives); S1 last
    headline_seeds=[0,1,2], ablation_seeds=[0],

    frac=(0.7,0.1,0.2), seed=42, min_test_pairs=200,

    chemberta_model="DeepChem/ChemBERTa-77M-MTR", token_max_len=96,
    morgan_bits=1024, morgan_radius=2,

    # target features
    target_min_freq=3,                  # keep proteins hitting >= this many drugs
    target_max=4096,
    coverage_gate=0.60,                 # HARD stop below this drug->target coverage

    # expanded feature dimensions (categorized targets, GO/Pfam, ATC, physchem, CYP hook)
    feature_version="v2multi",          # bump if you change what goes into Xmol/Xtgt -> avoids stale-cache reuse
    enable_categorized_targets=True,    # split target/enzyme/transporter/carrier into separate channels
    enable_go_pfam=True,                # GO + Pfam annotations of each drug's target proteins
    go_pfam_min_freq=3, go_pfam_max=2048,
    enable_atc=True,                    # WHO ATC classification: full code + level-1 anatomical group
    atc_min_freq=2,
    atc_source_url="https://raw.githubusercontent.com/dhimmel/drugbank/gh-pages/data/drugbank-slim.tsv",
    enable_physchem=True,               # RDKit MW/logP/TPSA/HBD/HBA/RotBonds, z-scored
    enable_cyp_roles=False,             # CYP substrate/inhibitor/inducer roles -- needs licensed DrugBank export
    cyp_role_csv=None,                  # set to a local CSV path if you have institutional access; see fetch_cyp_roles()
    run_feature_ablation_ladder=True,   # adds a 3rd variant: mol_only_classic (chemberta+morgan only -- the old baseline)

    hidden=256, dropout=0.3, pair_hidden=512,

    lr=1e-3, min_lr=1e-6, weight_decay=1e-2, batch_size=1024, eval_batch=4096,
    warmup_epochs=5, max_epochs=400, patience=60, min_delta=2e-4,
    min_epochs=40,        # never early-stop before this (let each run develop)
    ema_beta=0.8,         # smooth noisy val micro-AP for stop decision
    lr_factor=0.5, lr_patience=8, grad_clip=1.0,
    asl_gamma_neg=4.0, asl_gamma_pos=0.0, asl_clip=0.05,
    n_bootstrap=2000,
    FAST_MODE=False,   # False = full unattended run (default). True = ~10-min smoke test of the whole pipeline.
    workdir=os.path.expanduser("~/ddi_work"),
)
if CFG.FAST_MODE:
    CFG.max_epochs=12; CFG.patience=4; CFG.min_epochs=3; CFG.headline_seeds=[0]; CFG.n_bootstrap=500
import os; os.makedirs(CFG.workdir,exist_ok=True)
print("dataset",CFG.dataset,"| splits",CFG.splits,"| FAST",CFG.FAST_MODE)

dataset DrugBank | splits ['S2', 'S3', 'S1'] | FAST False


## 2. Environment

In [4]:
import importlib.util, importlib.metadata as _im, subprocess, sys, types
def _v(p):
    try: return _im.version(p)
    except Exception: return None
_NP0,_PD0=_v("numpy"),_v("pandas")
def _pip(a): subprocess.run([sys.executable,"-m","pip"]+a,check=False)
def ensure(m,pkg=None):
    if importlib.util.find_spec(m) is None: _pip(["install","-q",pkg or m])
ensure("rdkit","rdkit"); ensure("scipy","scipy"); ensure("requests","requests")
if importlib.util.find_spec("tdc") is None:
    _pip(["install","-q","--no-deps","PyTDC"])
    for d in ["fuzzywuzzy","dataclasses-json","yacs","huggingface_hub","requests","tqdm","openpyxl"]:
        ensure(d.split("-")[0].replace("_",""),d)
ensure("transformers","transformers")
for p,cur in [("numpy",_NP0),("pandas",_PD0)]:
    if cur and _v(p)!=cur: _pip(["install","-q",f"{p}=={cur}"])
from rdkit import RDLogger as _RDL; _RDL.DisableLog("rdApp.*")  # silence Morgan/parse spam
import numpy as np
if _NP0 and np.__version__!=_NP0:
    raise RuntimeError(f"numpy moved to {np.__version__}; Restart Session + Run All.")
import pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import rankdata
import json,time,math,random,warnings,gc,glob,io,requests,zipfile
warnings.filterwarnings("ignore")
for name,attr in [("tiledbsoma",None),("tdc.multi_pred.single_cell","CellXGeneTemplate"),
                  ("tdc.multi_pred.perturboutcome","PerturbOutcome")]:
    if name not in sys.modules:
        m=types.ModuleType(name)
        if attr: setattr(m,attr,object)
        sys.modules[name]=m
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def set_seed(s): random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)
print("torch",torch.__version__,"| numpy",np.__version__,"| device",DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 48.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 2.3 MB/s eta 0:00:00
torch 2.10.0+cu128 | numpy 2.0.2 | device cuda


## 3. TARGET DATA — fetch in code, join by DrugBank ID, and PROBE COVERAGE
DrugBank's protein export (target/enzyme/transporter/carrier) ends each protein row with
a semicolon-delimited list of the DrugBank drug IDs that hit it — a clean drug->target
map keyed to the same IDs DrugBank-DDI uses. We try the public export; if it needs auth
from Kaggle, we fall back to UniProt's DrugBank cross-references. The probe HARD-STOPS the
run if coverage is too low to help.

### Fix applied: enzyme/transporter/carrier stuck at 0% on Kaggle

Root cause: DrugBank's categorized protein export needs an account, so this always falls back to UniProt idmapping + reconstructing enzyme/transporter from each protein's EC number / GO annotation (`fetch_uniprot_ec_go`). That fallback fetch hits UniProt's `/uniprotkb/search` endpoint in ~100 small batched requests -- and Kaggle's outbound traffic shares a much more heavily-used IP pool than Colab's, so it gets rate-limited (HTTP 429) far more often there. When a chunk fails, everything in it silently falls back to the uncategorized 'target' bucket -- which is exactly the `enzyme=0%, transporter=0%, carrier=0%, target=94%` pattern you saw.

That alone would just mean a noisy run, but the real problem is caching: the old code wrote whatever it got -- even a totally empty result -- to `ec_go_class.json` / `drug_targets_by_cat.json`, and trusted that cache unconditionally on every later run. Your Kaggle "Section 0" cell copies a previous session's working directory forward from an attached input dataset, so once one Kaggle run got rate-limited, that empty result got carried into *every* subsequent session and was never retried. Colab starts clean each time, so it never hit this trap.

Fixed below: real retry/backoff that honors `Retry-After` on 429s, a proper `User-Agent` (UniProt throttles anonymous clients harder -- put your email in `_UNIPROT_HEADERS`), and -- the actual fix -- a degenerate (all-zero enzyme/transporter/carrier) result is never written to cache, and a degenerate cache already on disk is treated as stale and refetched rather than trusted.

**One-time cleanup needed:** your existing Kaggle working directory already has the poisoned cache files on disk from before this fix. Run this once before re-running the pipeline (safe -- it only deletes re-fetchable API caches, never checkpoints or results):
```python
import os
for f in ('drug_targets_by_cat.json', 'ec_go_class.json', 'drug_targets.json'):
    p = os.path.join(CFG.workdir, f)
    if os.path.exists(p):
        os.remove(p)
        print('removed', p)
```

In [5]:
DRUGBANK_PROTEIN_URLS = [
    # public "all proteins" CSV zips (target/enzyme/transporter/carrier)
    "https://go.drugbank.com/releases/latest/downloads/target-all-polypeptide-ids",
    "https://go.drugbank.com/releases/latest/downloads/enzyme-all-polypeptide-ids",
    "https://go.drugbank.com/releases/latest/downloads/transporter-all-polypeptide-ids",
    "https://go.drugbank.com/releases/latest/downloads/carrier-all-polypeptide-ids",
]
def _try_download(url, timeout=60):
    try:
        r=requests.get(url,timeout=timeout,headers={"User-Agent":"research"})
        if r.status_code==200 and len(r.content)>500: return r.content
    except Exception as e:
        print("   download failed:",url.split('/')[-1],e)
    return None

def _parse_protein_zip_or_csv(content):
    # DrugBank ships a zip containing one CSV; handle both
    text=None
    try:
        zf=zipfile.ZipFile(io.BytesIO(content))
        name=[n for n in zf.namelist() if n.lower().endswith(".csv")][0]
        text=zf.read(name).decode("utf-8","ignore")
    except Exception:
        try: text=content.decode("utf-8","ignore")
        except Exception: return {}
    df=pd.read_csv(io.StringIO(text))
    idcol="UniProt ID" if "UniProt ID" in df.columns else df.columns[0]
    dcol=[c for c in df.columns if c.lower().replace(" ","")=="drugids"]
    if not dcol: return {}
    dcol=dcol[0]; d2p={}
    for prot,ids in zip(df[idcol].astype(str).values, df[dcol].astype(str).values):
        if ids=="nan" or not ids.strip(): continue
        for d in (x.strip() for x in ids.split(";")):
            if d: d2p.setdefault(d,set()).add(str(prot))
    return d2p

def uniprot_idmap(drugbank_ids, chunk=100000):
    """Map DrugBank IDs -> UniProt accessions (a drug\'s protein targets) via UniProt\'s
    public ID-mapping REST API. No auth; works headless on Kaggle."""
    import time as _t
    base="https://rest.uniprot.org"
    d2p={}; ids=list(drugbank_ids)
    for i0 in range(0,len(ids),chunk):
        chunk_ids=ids[i0:i0+chunk]
        try:
            run=requests.post(f"{base}/idmapping/run",
                data={"from":"DrugBank","to":"UniProtKB","ids":",".join(chunk_ids)},timeout=60)
            job=run.json().get("jobId")
            if not job: print("   uniprot: no jobId",run.text[:200]); continue
        except Exception as e:
            print("   uniprot run failed:",e); continue
        # poll
        _poll_t0=_t.time()
        for _pi in range(60):
            try:
                st=requests.get(f"{base}/idmapping/status/{job}",timeout=60).json()
            except Exception: _t.sleep(3); continue
            if st.get("jobStatus") in ("RUNNING","NEW",None) and "results" not in st:
                if _pi%5==0: print(f"   uniprot idmapping: still running ({_t.time()-_poll_t0:.0f}s elapsed, poll {_pi+1}/60)")
                _t.sleep(3); continue
            break
        # fetch results (paginate via Link header), compact TSV: from<TAB>to
        url=f"{base}/idmapping/uniprotkb/results/{job}?format=tsv&fields=accession&size=500"
        while url:
            try: r=requests.get(url,timeout=120)
            except Exception as e: print("   uniprot results failed:",e); break
            lines=r.text.strip().split("\n")
            for ln in lines[1:]:
                parts=ln.split("\t")
                if len(parts)>=2 and parts[0] and parts[-1]:
                    d2p.setdefault(parts[0],set()).add(parts[-1])
            nxt=r.headers.get("Link","")
            url=nxt.split(";")[0].strip("<> ") if 'rel="next"' in nxt else None
    return d2p

def fetch_drug_targets(drug_ids=None, cache=f"{CFG.workdir}/drug_targets.json"):
    if os.path.exists(cache):
        d=json.load(open(cache)); print("   target cache hit"); return {k:set(v) for k,v in d.items()}
    merged={}
    # PRIMARY: UniProt ID-mapping (public API, works headless). Needs the drug ID list.
    if drug_ids is not None:
        try:
            merged=uniprot_idmap(drug_ids)
            print(f"   UniProt idmapping: {len(merged)} drugs with >=1 target")
        except Exception as e:
            print("   UniProt idmapping failed:",e); merged={}
    # FALLBACK: DrugBank public CSV export (often needs an account -> may be empty)
    if len(merged)==0:
        for url in DRUGBANK_PROTEIN_URLS:
            cc=_try_download(url)
            if cc is None: continue
            part=_parse_protein_zip_or_csv(cc)
            for d,ps in part.items(): merged.setdefault(d,set()).update(ps)
            print(f"   {url.split('/')[-1]}: +{len(part)} drugs")
    if merged:
        json.dump({k:sorted(v) for k,v in merged.items()},open(cache,"w"))
    return merged

def coverage_report(drug_ids, d2p):
    have=[d for d in drug_ids if len(d2p.get(d,()))>0]
    cnt=np.array([len(d2p.get(d,())) for d in drug_ids])
    prots=set().union(*[d2p[d] for d in have]) if have else set()
    return dict(coverage=len(have)/max(len(drug_ids),1), covered=len(have),
                n_drugs=len(drug_ids), median=float(np.median(cnt)),
                mean=float(cnt.mean()), distinct=len(prots))

def build_target_features(drug_ids, d2p, min_freq, max_t):
    freq={}
    for d in drug_ids:
        for p in d2p.get(d,()): freq[p]=freq.get(p,0)+1
    keep=[p for p,c in freq.items() if c>=min_freq]
    keep=sorted(keep,key=lambda p:-freq[p])[:max_t]; pidx={p:i for i,p in enumerate(keep)}
    X=np.zeros((len(drug_ids),len(keep)),np.float32)
    for i,d in enumerate(drug_ids):
        for p in d2p.get(d,()):
            if p in pidx: X[i,pidx[p]]=1.0
    return X, keep

# ================= UniProt-annotation reconstruction of target/enzyme/transporter =================
# When DrugBank's categorized exports aren't available (no account), UniProt idmapping still gives
# the drug->protein links, but as one undifferentiated bucket. DDIs are overwhelmingly PK-driven
# (CYP450 enzymes, P-gp/OATP transporters), not just primary-target overlap, so recovering the role
# matters for the mechanism-attribution story. UniProt's own per-protein annotation carries this:
# a catalytic EC number means enzyme; a transporter-activity GO annotation means transporter;
# otherwise it's left as an (uncategorized) target. This is an approximation of DrugBank's curated
# categories (which reflect pharmacological intent, not just molecular function) -- described as
# such in the paper -- not a reproduction of them. Carrier proteins have no comparable public
# functional signature, so that channel stays empty under this fallback (honest gap, not guessed).
_TRANSPORTER_GO_KEYWORDS = ("transporter activity", "transmembrane transporter")

# NOTE: Kaggle's outbound requests share a much more heavily-used IP pool than Colab's,
# so UniProt's public search endpoint rate-limits (HTTP 429) it far more aggressively.
# A generic User-Agent makes this worse -- UniProt's own guidance is to identify your
# client. Put a real contact email below; it costs nothing and measurably reduces throttling.
_UNIPROT_HEADERS = {"User-Agent": "TARGET-DDI-research/1.0 (contact: YOUR_EMAIL_HERE@example.com)"}

def fetch_uniprot_ec_go(protein_ids, chunk=20, cache=None, min_coverage=0.5):
    """UniProt accession -> (ec_numbers:set[str], go_text:str). Same batched-query pattern as
    fetch_go_pfam (OR-ed accessions, chunk<=~25 to stay under UniProt's URL length cap), but
    pulling the 'ec' and 'go' fields instead of go_id/xref_pfam -- 'go' returns full GO term
    names (not just IDs), which is what the transporter keyword match below needs.

    FIX: this used to unconditionally cache whatever it got, including a near-empty result
    from a run that got rate-limited on every chunk -- and unconditionally trust that cache
    on the next run. On Kaggle (shared IP pool -> heavier UniProt throttling than Colab,
    especially with no identifying User-Agent) that meant one bad run poisoned every future
    run permanently, silently, with enzyme/transporter stuck at 0%. Now: real backoff
    (honors Retry-After on 429), a real User-Agent, per-chunk failure counts printed as they
    happen (not just every 20th), and a cache is only trusted/written if it actually covers
    a useful fraction of the requested proteins -- a degenerate result is never persisted
    and a degenerate cache on disk is treated as stale and refetched instead of returned."""
    cache=cache or f"{CFG.workdir}/ec_go_class.json"
    protein_ids=set(protein_ids)
    if os.path.exists(cache):
        d=json.load(open(cache))
        cached={p:(set(v["ec"]),v["go"]) for p,v in d.items()}
        cov=len(protein_ids & cached.keys())/max(len(protein_ids),1)
        if cov>=min_coverage:
            print(f"   EC/GO classification cache hit ({len(cached)} proteins, covers {cov*100:.0f}% of current set)")
            return cached
        print(f"   EC/GO cache on disk only covers {cov*100:.0f}% of the current protein set "
              f"({len(cached)} cached vs {len(protein_ids)} needed) -- treating it as stale from a "
              f"failed/rate-limited prior run and refetching instead of trusting it")
    ids=sorted(protein_ids); out={}
    print(f"   fetching EC/GO for {len(ids)} proteins (enzyme/transporter reconstruction)...")
    t0=time.time(); n_fail=0
    for i0 in range(0,len(ids),chunk):
        part=ids[i0:i0+chunk]
        q=" OR ".join(f"accession:{p}" for p in part)
        r=None
        for _try in range(5):
            try:
                r=requests.get("https://rest.uniprot.org/uniprotkb/search",
                    params={"query":q,"fields":"accession,ec,go","format":"tsv","size":max(chunk,50)},
                    headers=_UNIPROT_HEADERS, timeout=90)
                if r.status_code==200: break
                if r.status_code==429:
                    wait=float(r.headers.get("Retry-After", 5*(_try+1)))
                    print(f"   EC/GO chunk {i0//chunk}: rate-limited (429), waiting {wait:.0f}s (attempt {_try+1}/5)")
                    time.sleep(wait); continue
                time.sleep(1.5*(_try+1))
            except Exception as e:
                print(f"   EC/GO chunk {i0//chunk}: request error ({e}), retrying (attempt {_try+1}/5)")
                time.sleep(1.5*(_try+1))
        if r is None or r.status_code!=200:
            n_fail+=1
            print(f"   EC/GO chunk {i0//chunk}: giving up after retries, HTTP {getattr(r,'status_code','no response')}")
            continue
        try:
            lines=r.text.strip().split("\n")
            if len(lines)<2: continue
            for ln in lines[1:]:
                cols=ln.split("\t")
                if not cols or not cols[0]: continue
                acc=cols[0]
                ec=set(x.strip() for x in cols[1].split(";") if x.strip()) if len(cols)>1 else set()
                go=cols[2].strip() if len(cols)>2 else ""
                out[acc]=(ec,go)
        except Exception as e:
            print(f"   EC/GO chunk {i0//chunk} parse failed:",e); continue
        if (i0//chunk)%5==0:
            print(f"   EC/GO: {min(i0+chunk,len(ids))}/{len(ids)} proteins ({time.time()-t0:.0f}s elapsed, {n_fail} chunk failures so far)")
    coverage=len(out)/max(len(ids),1)
    print(f"   EC/GO: {len(out)}/{len(ids)} proteins annotated ({coverage*100:.0f}% coverage, {n_fail} chunk failures)")
    if coverage>=min_coverage:
        json.dump({p:{"ec":sorted(ec),"go":go} for p,(ec,go) in out.items()},open(cache,"w"))
    else:
        print(f"   EC/GO coverage below {min_coverage*100:.0f}% -- NOT writing this to cache, so the next "
              f"run retries fresh instead of reusing today's bad/rate-limited result")
    return out

def classify_by_annotation(d2p, ann):
    """drug->protein map (uncategorized) + protein->(ec,go_text) annotation -> reconstructed
    target/enzyme/transporter dicts. A protein with a catalytic EC number is an enzyme; else a
    transporter-activity GO term makes it a transporter; else it stays an (uncategorized) target."""
    cats={"target":{}, "enzyme":{}, "transporter":{}}
    for d,prots in d2p.items():
        for p in prots:
            ec,go_text=ann.get(p,(set(),""))
            if ec:
                cat="enzyme"
            elif any(kw in go_text.lower() for kw in _TRANSPORTER_GO_KEYWORDS):
                cat="transporter"
            else:
                cat="target"
            cats[cat].setdefault(d,set()).add(p)
    return cats

def fetch_drug_targets_by_category(drug_ids=None, cache=f"{CFG.workdir}/drug_targets_by_cat.json"):
    """drug -> protein map, split into target / enzyme / transporter / carrier channels.
    Primary source: DrugBank's own categorized CSV exports (DRUGBANK_PROTEIN_URLS), which are
    properly split by category. Those often need a DrugBank account, so if all four fail we fall
    back to UniProt idmapping for the drug->protein links, then RECONSTRUCT enzyme/transporter/
    target from each protein's own UniProt annotation (EC number, transporter-activity GO term)
    instead of leaving everything in one undifferentiated 'target' bucket -- see
    fetch_uniprot_ec_go / classify_by_annotation above.

    FIX: this is the actual cause of enzyme/transporter/carrier reading 0% on Kaggle while
    Colab works. This cache used to be trusted unconditionally. Your Kaggle "Section 0" cell
    copies a previous session's working directory forward from an attached input dataset --
    which means if EC/GO annotation ever got rate-limited on a past Kaggle run (see the note
    in fetch_uniprot_ec_go: Kaggle's shared IP pool gets throttled by UniProt much harder than
    Colab's), that run's degenerate all-'target' categorization got written to
    drug_targets_by_cat.json once, and every subsequent Kaggle session -- including this one --
    has been silently loading that same poisoned cache forever instead of refetching. Colab has
    no such carried-forward cache, so it fetches clean each time and works. Fix: a cache is only
    trusted if it actually has enzyme/transporter/carrier signal (or genuinely has no drug_ids
    to check against); a degenerate cache is treated as stale and refetched, and a degenerate
    *new* result is not written back to disk, so a bad run no longer poisons the next one."""
    if os.path.exists(cache):
        d=json.load(open(cache))
        cats_cached={cat:{k:set(v) for k,v in dd.items()} for cat,dd in d.items()}
        n_tgt=len(cats_cached.get("target",{})); n_enz=len(cats_cached.get("enzyme",{}))
        n_trans=len(cats_cached.get("transporter",{})); n_car=len(cats_cached.get("carrier",{}))
        if (n_enz+n_trans+n_car)>0 or n_tgt==0:
            print(f"   categorized target cache hit (target={n_tgt}, enzyme={n_enz}, transporter={n_trans}, carrier={n_car})")
            return cats_cached
        print(f"   categorized target cache on disk has target={n_tgt} but enzyme=transporter=carrier=0 -- "
              f"this is the signature of a prior run whose EC/GO fetch got rate-limited (common on Kaggle's "
              f"shared IPs, carried forward by the Section-0 resume cell). Ignoring this cache and refetching.")
    cats={"target":{}, "enzyme":{}, "transporter":{}, "carrier":{}}
    got_categorized=False
    for url in DRUGBANK_PROTEIN_URLS:
        cat=url.split("/")[-1].split("-")[0]
        cc=_try_download(url)
        if cc is None: continue
        part=_parse_protein_zip_or_csv(cc)
        if part:
            cats[cat]=part; got_categorized=True
            print(f"   {cat}: {len(part)} drugs (DrugBank categorized export)")
    if not got_categorized:
        print("   DrugBank categorized exports unavailable (likely needs an account) -> falling back to"
              " UniProt idmapping + EC/GO-based enzyme/transporter/target reconstruction"
              " (carrier has no comparable public signature, so that channel stays empty)")
        if drug_ids is not None:
            try:
                merged=uniprot_idmap(drug_ids)
                print(f"   UniProt idmapping: {len(merged)} drugs with >=1 target (uncategorized)")
                all_prots=set().union(*merged.values()) if merged else set()
                ann=fetch_uniprot_ec_go(all_prots)
                if len(ann)==0 and len(all_prots)>0:
                    print("   WARNING: EC/GO annotation came back empty this run -- enzyme/transporter will be"
                          " 0 this time (everything falls into 'target'). Likely UniProt rate-limiting; this"
                          " result will NOT be cached, so simply re-running this cell later should recover it.")
                reclass=classify_by_annotation(merged, ann)
                for cat in ("target","enzyme","transporter"):
                    cats[cat]=reclass.get(cat,{})
                    print(f"   [{cat}] reconstructed: {len(cats[cat])} drugs")
            except Exception as e:
                print("   UniProt idmapping / classification failed:",e)
    got_signal = len(cats["enzyme"])>0 or len(cats["transporter"])>0 or len(cats["carrier"])>0
    if got_signal or not drug_ids:
        json.dump({cat:{k:sorted(v) for k,v in dd.items()} for cat,dd in cats.items()},open(cache,"w"))
    else:
        print("   enzyme/transporter/carrier all empty this run -- NOT writing to cache, so the next run"
              " retries the EC/GO fetch instead of being permanently stuck on this result.")
    return cats

def fetch_go_pfam(protein_ids, chunk=20, cache=f"{CFG.workdir}/go_pfam.json"):  # chunk<=~25 avoids UniProt HTTP 400
    """UniProt accession -> (GO term set, Pfam domain set), via UniProt's search endpoint,
    batched by OR-ing accessions into one query per chunk. Best-effort: a failed chunk is
    skipped (those proteins just contribute no GO/Pfam signal), never hard-fails the run."""
    if os.path.exists(cache):
        d=json.load(open(cache)); print("   GO/Pfam cache hit")
        return {p:(set(v["go"]),set(v["pfam"])) for p,v in d.items()}
    ids=sorted(protein_ids); out={}
    print(f"   fetching GO/Pfam for {len(ids)} proteins in chunks of {chunk}...")
    t0=time.time()
    for i0 in range(0,len(ids),chunk):
        part=ids[i0:i0+chunk]
        q=" OR ".join(f"accession:{p}" for p in part)     # <=20 accessions -> under UniProt length cap
        r=None
        for _try in range(3):
            try:
                r=requests.get("https://rest.uniprot.org/uniprotkb/search",
                    params={"query":q,"fields":"accession,go_id,xref_pfam","format":"tsv","size":max(chunk,50)},
                    headers=_UNIPROT_HEADERS, timeout=90)
                if r.status_code==200: break
            except Exception:
                import time as _t; _t.sleep(1.0)
        if r is None or r.status_code!=200:
            if (i0//chunk)%20==0: print(f"   GO/Pfam chunk {i0//chunk}: HTTP {getattr(r,'status_code','err')}")
            continue
        try:
            lines=r.text.strip().split("\n")
            if len(lines)<2: continue
            for ln in lines[1:]:
                cols=ln.split("\t")
                if not cols or not cols[0]: continue
                acc=cols[0]
                go=set(x.strip() for x in cols[1].split(";") if x.strip()) if len(cols)>1 else set()
                pf=set(x.strip().rstrip(";") for x in cols[2].split(";") if x.strip()) if len(cols)>2 else set()
                out[acc]=(go,pf)
        except Exception as e:
            print(f"   GO/Pfam chunk {i0//chunk} failed:",e); continue
        if (i0//chunk)%5==0:
            print(f"   GO/Pfam: {min(i0+chunk,len(ids))}/{len(ids)} proteins ({time.time()-t0:.0f}s elapsed)")
    json.dump({p:{"go":sorted(g),"pfam":sorted(f)} for p,(g,f) in out.items()},open(cache,"w"))
    return out

def build_go_pfam_features(drug_ids, d2p_union, prot2gopfam, min_freq, max_t):
    """Drug-level GO/Pfam multi-hot = union of GO terms / Pfam domains over all of that
    drug's target/enzyme/transporter/carrier proteins. Lets the model generalize across
    functionally-similar proteins even when it never saw that exact UniProt ID in training
    -- the direct fix for the cold-start blind spot of the raw binary target indicator."""
    go_freq={}; pf_freq={}; d2go={}; d2pf={}
    for d in drug_ids:
        gos=set(); pfs=set()
        for p in d2p_union.get(d,()):
            g,f=prot2gopfam.get(p,(set(),set())); gos|=g; pfs|=f
        d2go[d]=gos; d2pf[d]=pfs
        for g in gos: go_freq[g]=go_freq.get(g,0)+1
        for f in pfs: pf_freq[f]=pf_freq.get(f,0)+1
    keep_go=sorted([g for g,c in go_freq.items() if c>=min_freq],key=lambda g:-go_freq[g])[:max_t]
    keep_pf=sorted([f for f,c in pf_freq.items() if c>=min_freq],key=lambda f:-pf_freq[f])[:max_t]
    gidx={g:i for i,g in enumerate(keep_go)}; pidx={f:i for i,f in enumerate(keep_pf)}
    Xgo=np.zeros((len(drug_ids),len(keep_go)),np.float32)
    Xpf=np.zeros((len(drug_ids),len(keep_pf)),np.float32)
    for i,d in enumerate(drug_ids):
        for g in d2go[d]:
            if g in gidx: Xgo[i,gidx[g]]=1.0
        for f in d2pf[d]:
            if f in pidx: Xpf[i,pidx[f]]=1.0
    return Xgo,Xpf,keep_go,keep_pf

def fetch_atc(drug_ids, url=None, cache=f"{CFG.workdir}/atc.json"):
    """DrugBank ID -> set(ATC codes), from dhimmel/drugbank's public open-data export
    (a DrugBank 4.2-derived TSV on GitHub). ATC assignments are essentially static, so
    this older snapshot is still a fine approximation for approved small molecules."""
    url=url or CFG.atc_source_url
    if os.path.exists(cache):
        d=json.load(open(cache)); print("   ATC cache hit")
        return {k:set(v) for k,v in d.items()}
    d2atc={}
    try:
        r=requests.get(url,timeout=60)
        if r.status_code==200:
            df=pd.read_csv(io.StringIO(r.text),sep="\t",dtype=str)
            idc=[c for c in df.columns if c.lower()=="drugbank_id"][0]
            atc_c=[c for c in df.columns if c.lower()=="atc_codes"][0]
            wanted=set(drug_ids)
            for did,codes in zip(df[idc],df[atc_c]):
                if did not in wanted or not isinstance(codes,str) or not codes.strip(): continue
                cs={c.strip() for c in codes.replace("|",";").split(";") if c.strip()}
                if cs: d2atc[did]=cs
            print(f"   ATC: {len(d2atc)}/{len(drug_ids)} drugs matched")
        else:
            print(f"   ATC fetch: HTTP {r.status_code}, skipping ATC feature")
    except Exception as e:
        print("   ATC fetch failed:",e,"-- skipping ATC feature")
    json.dump({k:sorted(v) for k,v in d2atc.items()},open(cache,"w"))
    return d2atc

def build_atc_features(drug_ids, d2atc, min_freq):
    """Two blocks concatenated: full ATC code (filtered by min_freq, like target proteins)
    and the coarser level-1 anatomical-group letter (dense, ~14-dim, no filtering needed)."""
    full_freq={}; l1_freq={}; d2full={}; d2l1={}
    for d in drug_ids:
        codes=d2atc.get(d,set())
        d2full[d]=codes; d2l1[d]={c[0] for c in codes if c}
        for c in codes: full_freq[c]=full_freq.get(c,0)+1
        for c in d2l1[d]: l1_freq[c]=l1_freq.get(c,0)+1
    keep_full=sorted([c for c,n in full_freq.items() if n>=min_freq],key=lambda c:-full_freq[c])
    keep_l1=sorted(l1_freq.keys())
    fidx={c:i for i,c in enumerate(keep_full)}; lidx={c:i for i,c in enumerate(keep_l1)}
    Xfull=np.zeros((len(drug_ids),len(keep_full)),np.float32)
    Xl1=np.zeros((len(drug_ids),len(keep_l1)),np.float32)
    for i,d in enumerate(drug_ids):
        for c in d2full[d]:
            if c in fidx: Xfull[i,fidx[c]]=1.0
        for c in d2l1[d]:
            if c in lidx: Xl1[i,lidx[c]]=1.0
    return np.concatenate([Xfull,Xl1],1).astype(np.float32), keep_full, keep_l1

def fetch_cyp_roles(drug_ids, csv_path=None):
    """CYP substrate/inhibitor/inducer roles are the most mechanistically precise PK-DDI
    feature, but DrugBank only exposes the *association* (drug hits enzyme X) for free --
    the specific *role* (substrate vs inhibitor vs inducer) is behind DrugBank's academic/
    commercial license. This is a pluggable hook, not a scraper: if you have that export
    (e.g. via institutional access), point CFG.cyp_role_csv at a CSV with columns
    [drugbank_id, cyp, role] and this picks it up automatically. Without it, it returns a
    zero-width feature and the run proceeds -- it never fakes the signal."""
    csv_path=csv_path or CFG.cyp_role_csv
    if not csv_path or not os.path.exists(csv_path):
        print("   CYP roles: no licensed export configured (CFG.cyp_role_csv) -- skipping this feature")
        return np.zeros((len(drug_ids),0),np.float32), []
    df=pd.read_csv(csv_path,dtype=str)
    combo=sorted(set(df["cyp"]+":"+df["role"]))
    cidx={c:i for i,c in enumerate(combo)}
    d2c={}
    for did,cyp,role in zip(df["drugbank_id"],df["cyp"],df["role"]):
        d2c.setdefault(did,set()).add(f"{cyp}:{role}")
    X=np.zeros((len(drug_ids),len(combo)),np.float32)
    for i,d in enumerate(drug_ids):
        for c in d2c.get(d,()):
            if c in cidx: X[i,cidx[c]]=1.0
    print(f"   CYP roles: loaded {len(combo)} (cyp,role) combos from {csv_path}")
    return X, combo

# ================= ESM-2 PROTEIN EMBEDDINGS (mechanistic feature) =================
# An unseen drug's target may be a NOVEL protein, but ESM-2 places it near functionally
# similar known proteins; pooling a drug's targets' embeddings gives a cold-start-robust
# target representation that binary presence vectors cannot. Verified: on similar-but-novel
# targets, pooled ESM-2 >> binary. Fetched per-accession from UniProt's embeddings API, cached.
def fetch_esm_embeddings(uniprot_ids, cache=None):
    cache=cache or f"{CFG.workdir}/esm_emb.npz"
    if os.path.exists(cache):
        d=np.load(cache,allow_pickle=True)
        emb={k:d["emb"][i] for i,k in enumerate(d["ids"])}
        print(f"   ESM-2 cache hit ({len(emb)} proteins)"); return emb
    emb={}; ids=list(uniprot_ids)
    for n,acc in enumerate(ids):
        try:
            r=requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}/embeddings",headers=_UNIPROT_HEADERS,timeout=30)
            if r.status_code==200:
                j=r.json(); v=j.get("embedding") or j.get("value")
                if v: emb[acc]=np.asarray(v,np.float32)
        except Exception: pass
        if n%200==0 and n>0: print(f"   ESM-2: {len(emb)}/{n} fetched ...")
    if emb:
        keys=list(emb); np.savez_compressed(cache,ids=np.array(keys),emb=np.stack([emb[k] for k in keys]))
    print(f"   ESM-2: {len(emb)}/{len(ids)} proteins embedded"); return emb

def build_esm_drug_features(drug_ids, d2p, emb):
    if not emb: return np.zeros((len(drug_ids),0),np.float32), False
    dim=len(next(iter(emb.values()))); X=np.zeros((len(drug_ids),dim),np.float32)
    for i,d in enumerate(drug_ids):
        vecs=[emb[p] for p in d2p.get(d,()) if p in emb]
        if vecs: X[i]=np.mean(vecs,0)
    mu,sd=X.mean(0),X.std(0); sd[sd==0]=1
    return ((X-mu)/sd).astype(np.float32), True


## 4. Splits (S1/S2/S3), leakage audit, tie-correct metrics

In [6]:
def canonical(a,b): return (a,b) if str(a)<=str(b) else (b,a)
def build_pair_table(df,id1,id2,s1,s2,lab):
    ds={}
    for _,r in df[[id1,s1]].drop_duplicates(id1).iterrows(): ds[r[id1]]=r[s1]
    for _,r in df[[id2,s2]].drop_duplicates(id2).iterrows(): ds.setdefault(r[id2],r[s2])
    classes=sorted(pd.unique(df[lab]).tolist()); c2i={c:i for i,c in enumerate(classes)}
    pl={}
    for a,b,l in zip(df[id1].values,df[id2].values,df[lab].values):
        pl.setdefault(canonical(a,b),set()).add(c2i[l])
    rows=[{"u":k[0],"v":k[1],"y":sorted(v)} for k,v in pl.items()]
    return pd.DataFrame(rows), ds, len(classes)

def split_S1(pairs,frac,seed):
    rng=np.random.default_rng(seed); idx=rng.permutation(len(pairs)); n=len(pairs)
    a=int(frac[0]*n); b=a+int(frac[1]*n)
    return {"train":pairs.iloc[idx[:a]].reset_index(drop=True),
            "valid":pairs.iloc[idx[a:b]].reset_index(drop=True),
            "test":pairs.iloc[idx[b:]].reset_index(drop=True)},{"retention":1.0}

def split_entity(pairs,mode,frac,seed):
    rng=np.random.default_rng(seed); drugs=rng.permutation(pd.unique(pairs[["u","v"]].values.ravel()))
    n=len(drugs);a=int(frac[0]*n);b=a+int(frac[1]*n);tr_d,va_d,te_d=set(drugs[:a]),set(drugs[a:b]),set(drugs[b:])
    # train = pairs with BOTH endpoints assigned to train
    both=pairs.apply(lambda r: r["u"] in tr_d and r["v"] in tr_d, axis=1)
    train=pairs[both]
    # "seen" = drugs that actually appear in a TRAIN PAIR (what the model can learn),
    # which is exactly what audit() recomputes -> keeps split and audit consistent.
    seen=set(train["u"])|set(train["v"])
    need_unseen=1 if mode=="S2" else 2      # test/valid pair unseen-endpoint count
    def bucket(r):
        u,v=r["u"],r["v"]
        if u in seen and v in seen: return "train" if (u in tr_d and v in tr_d) else "drop"
        un=[d for d in (u,v) if d not in seen]
        if len(un)!=need_unseen: return "drop"
        in_va=all(d in va_d for d in un); in_te=all(d in te_d for d in un)
        if in_va and not in_te: return "valid"
        if in_te and not in_va: return "test"
        return "drop"
    bk=pairs.apply(bucket,axis=1)
    return {s:pairs[bk==s].reset_index(drop=True) for s in ["train","valid","test"]},\
           {"retention":float(bk.isin(["train","valid","test"]).mean())}

def make_split(pairs,mode,frac,seed):
    return split_S1(pairs,frac,seed) if mode=="S1" else split_entity(pairs,mode,frac,seed)

def audit(sp,mode,id2n=None,ei=None):
    ks={s:set(zip(sp[s]["u"],sp[s]["v"])) for s in ["train","valid","test"]}
    assert ks["train"].isdisjoint(ks["valid"]) and ks["train"].isdisjoint(ks["test"]) and ks["valid"].isdisjoint(ks["test"]),"pair overlap"
    tr_d=set(sp["train"]["u"])|set(sp["train"]["v"])
    if mode!="S1":
        want=1 if mode=="S2" else 2
        for u,v in ks["test"]: assert (u not in tr_d)+(v not in tr_d)==want,f"{mode} unseen-count"
    if ei is not None:
        es=set(map(tuple,ei.t().tolist())); tr_nodes={id2n[d] for d in tr_d}; leak=0
        for s in ["valid","test"]:
            for u,v in zip(sp[s]["u"],sp[s]["v"]):
                iu,iv=id2n[u],id2n[v]
                if (iu,iv) in es or (iv,iu) in es: leak+=1
        assert leak==0,f"{leak} held-out edges in graph"
        bad=sum(1 for s2,d in zip(ei[0].tolist(),ei[1].tolist()) if d in tr_nodes and s2 not in tr_nodes)
        assert bad==0,f"{bad} unseen->train edges"
    print(f"  [audit OK] {mode}: train={len(sp['train'])} valid={len(sp['valid'])} test={len(sp['test'])}")
    if len(sp["test"])<CFG.min_test_pairs: print(f"  WARN {len(sp['test'])} test pairs (<{CFG.min_test_pairs}); high variance")

def multihot(ys,C):
    Y=np.zeros((len(ys),C),np.float32)
    for i,l in enumerate(ys):
        for k in l: Y[i,k]=1.0
    return Y
def _auroc_cols(y,p):
    y=y.astype(bool);N,C=y.shape;out=np.full(C,np.nan)
    R=np.apply_along_axis(lambda c:rankdata(c,method="average"),0,p)
    npos=y.sum(0).astype(float);nneg=N-npos;ok=(npos>0)&(nneg>0);sr=(R*y).sum(0)
    with np.errstate(all="ignore"): out[ok]=((sr-npos*(npos+1)/2)/(npos*nneg))[ok]
    return out
def _ap_cols(y,p):
    from sklearn.metrics import average_precision_score
    N,C=y.shape;out=np.full(C,np.nan)
    for c in range(C):
        s=int(y[:,c].sum())
        if 0<s<N: out[c]=average_precision_score(y[:,c],p[:,c])
    return out
def _ap_micro(y,p):
    y=y.ravel().astype(np.float64);p=p.ravel()
    o=np.argsort(-p,kind="mergesort");ys=y[o];ps=p[o]
    tp=np.cumsum(ys);fp=np.cumsum(1-ys);P=tp[-1]
    if P==0: return float("nan")
    last=np.r_[np.diff(ps)!=0,True];tpg=tp[last];fpg=fp[last]
    prec=tpg/(tpg+fpg);rec=tpg/P;prev=np.r_[0.0,rec[:-1]]
    return float(np.sum((rec-prev)*prec))
def tune_thresholds(yv,pv):
    grid=np.linspace(0.05,0.95,19);y=yv.astype(bool);C=y.shape[1]
    best=np.zeros(C);thr=np.full(C,0.5,np.float32);has=y.any(0)
    for t in grid:
        pr=pv>=t;tp=(pr&y).sum(0,dtype=np.float64);fp=(pr&~y).sum(0,dtype=np.float64);fn=(~pr&y).sum(0,dtype=np.float64)
        den=2*tp+fp+fn;f1=np.where(den>0,2*tp/np.where(den>0,den,1),0.0)
        imp=has&(f1>best);best[imp]=f1[imp];thr[imp]=t
    return thr
def evaluate(y,p,thr):
    from sklearn.metrics import f1_score
    yh=(p>=thr[None,:]).astype(int);o={}
    o["micro_f1"]=float(f1_score(y,yh,average="micro",zero_division=0))
    o["macro_f1"]=float(f1_score(y,yh,average="macro",zero_division=0))
    au=_auroc_cols(y,p);o["auroc_per_type"]=float(np.nanmean(au)) if np.any(~np.isnan(au)) else float("nan")
    o["auprc_per_type"]=float(np.nanmean(_ap_cols(y,p)))
    o["micro_auprc"]=_ap_micro(y,p)
    prev=y.mean();o["pr_lift"]=float(o["micro_auprc"]/prev) if prev>0 else float("nan")
    for k in (1,3,5):
        ok=np.argsort(-p,axis=1)[:,:k];hit=tot=0
        for i in range(len(y)):
            tr=set(np.where(y[i]==1)[0])
            if not tr: continue
            tot+=1;hit+=1 if tr&set(ok[i].tolist()) else 0
        o[f"top{k}"]=hit/max(tot,1)
    return o
def paired_bootstrap(y,pf,pb,nb,seed=0):
    rng=np.random.default_rng(seed);N=len(y);d=np.empty(nb)
    print(f"    [bootstrap] N={N} pairs x C={y.shape[1]} classes, nb={nb} resamples -- timing first 10 iters...")
    t0=time.time();report_every=max(1,nb//20)
    for b in range(nb):
        idx=rng.integers(0,N,N);d[b]=_ap_micro(y[idx],pf[idx])-_ap_micro(y[idx],pb[idx])
        if b==9:
            per_iter=(time.time()-t0)/10
            print(f"    [bootstrap] ~{per_iter*1000:.0f}ms/iter -> est. total {per_iter*nb/60:.1f} min for nb={nb}")
        if (b+1)%report_every==0 or b+1==nb:
            elapsed=time.time()-t0;frac=(b+1)/nb;eta=elapsed/frac-elapsed
            print(f"    [bootstrap] {b+1}/{nb} ({frac*100:.0f}%) elapsed={elapsed:.0f}s  eta={eta:.0f}s")
    print(f"    [bootstrap] done in {time.time()-t0:.0f}s")
    lo,hi=np.percentile(d,[2.5,97.5]);return float(_ap_micro(y,pf)-_ap_micro(y,pb)),float(lo),float(hi)

## 5. Molecular features (ChemBERTa + Morgan), cached

In [7]:
from tdc.multi_pred import DDI
from rdkit import Chem
from rdkit.Chem import AllChem
def load_dataset(name):
    d=DDI(name=name);df=d.get_data();c={k.lower():k for k in df.columns}
    id1=c.get("drug1_id","Drug1_ID");id2=c.get("drug2_id","Drug2_ID")
    s1=c.get("drug1","Drug1");s2=c.get("drug2","Drug2");lab=c.get("y","Y")
    df=df[[id1,s1,id2,s2,lab]].dropna()
    if df[lab].apply(lambda v:isinstance(v,(list,tuple,np.ndarray))).any(): df=df.explode(lab)
    return build_pair_table(df,id1,id2,s1,s2,lab)
def chemberta(smiles,cache):
    if os.path.exists(cache):
        E=np.load(cache)
        if len(E)==len(smiles): print("   ChemBERTa cache hit"); return E
    from transformers import AutoTokenizer,AutoModel
    tk=AutoTokenizer.from_pretrained(CFG.chemberta_model);m=AutoModel.from_pretrained(CFG.chemberta_model).to(DEVICE).eval()
    out=[];n=len(smiles);nb_batches=math.ceil(n/64);t0=time.time()
    print(f"   ChemBERTa: embedding {n} molecules in {nb_batches} batches on {DEVICE}...")
    with torch.no_grad():
        for bi,i in enumerate(range(0,n,64)):
            e=tk(smiles[i:i+64],padding=True,truncation=True,max_length=CFG.token_max_len,return_tensors="pt").to(DEVICE)
            h=m(**e).last_hidden_state;mask=e["attention_mask"].unsqueeze(-1).float()
            out.append(((h*mask).sum(1)/mask.sum(1).clamp(min=1)).cpu().numpy())
            if (bi+1)%max(1,nb_batches//20)==0 or bi+1==nb_batches:
                elapsed=time.time()-t0;frac=(bi+1)/nb_batches;eta=elapsed/frac-elapsed
                print(f"   ChemBERTa: batch {bi+1}/{nb_batches} ({frac*100:.0f}%) elapsed={elapsed:.0f}s eta={eta:.0f}s")
    E=np.concatenate(out).astype(np.float32);np.save(cache,E);del m;torch.cuda.empty_cache();return E
def morgan(smiles):
    from rdkit.Chem import rdFingerprintGenerator
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=CFG.morgan_radius,fpSize=CFG.morgan_bits)
    X=np.zeros((len(smiles),CFG.morgan_bits),np.float32); bad=0
    for i,s in enumerate(smiles):
        mm=Chem.MolFromSmiles(s)
        if mm is None: bad+=1; continue          # unparseable SMILES -> zero row (kept, handled downstream)
        X[i]=np.array(gen.GetFingerprint(mm),np.float32)
    if bad: print(f"   note: {bad} SMILES unparseable -> zero fingerprint (handled)")
    return X
def molecular_features(drug_ids,ds):
    sm=[ds[d] for d in drug_ids]
    try: cb=chemberta(sm,f"{CFG.workdir}/emb_{CFG.dataset}.npy")
    except Exception as e: print("   ChemBERTa failed:",e); cb=np.zeros((len(sm),0),np.float32)
    return np.concatenate(([cb] if cb.shape[1] else [])+[morgan(sm)],1).astype(np.float32)

def physchem_features(smiles):
    """6 cheap RDKit descriptors (MW, logP, TPSA, HBD, HBA, rotatable bonds), z-scored.
    Free -- reuses RDKit already loaded for Morgan fingerprints, no network call."""
    from rdkit.Chem import Descriptors
    X=np.zeros((len(smiles),6),np.float32); bad=0
    for i,s in enumerate(smiles):
        mm=Chem.MolFromSmiles(s)
        if mm is None: bad+=1; continue
        X[i]=[Descriptors.MolWt(mm),Descriptors.MolLogP(mm),Descriptors.TPSA(mm),
              Descriptors.NumHDonors(mm),Descriptors.NumHAcceptors(mm),Descriptors.NumRotatableBonds(mm)]
    if bad: print(f"   note: {bad} SMILES unparseable for physchem -> zero row (handled)")
    mu=X.mean(0,keepdims=True); sd=X.std(0,keepdims=True); sd[sd<1e-6]=1.0
    return ((X-mu)/sd).astype(np.float32)


## 6. Model — molecular + target encoders with per-prediction gated fusion
The fusion lesson from validation: a zero-init gate strangles the target signal, and a
fixed blend dilutes it. Instead the gate is conditioned on **target availability** (how
many targets each drug has) so the model routes to targets when they exist and to
molecular features when they do not. `add_target=False` cleanly removes the branch.

In [8]:
class Enc(nn.Module):
    def __init__(s,din,h,drop):
        super().__init__(); s.net=nn.Sequential(nn.Linear(din,h),nn.LayerNorm(h),nn.ReLU(),nn.Dropout(drop),
                          nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU(),nn.Dropout(drop))
    def forward(s,x): return s.net(x)

class SharedTargetLayer(nn.Module):
    """THE EDGE: shared-target message passing over a drug<->protein bipartite graph.
    An unseen drug is enriched by the TRAINING drugs it shares targets with; only
    training-drug embeddings are sources (train_mask) so no test leak. Target
    annotations are external biology, never learned from DDI labels."""
    def __init__(s,h): super().__init__(); s.lin=nn.Linear(h,h); s.norm=nn.LayerNorm(h)
    def forward(s,h,tgt_bin,train_mask):
        Ttr=tgt_bin*train_mask.unsqueeze(1).to(h.dtype)
        tdeg=Ttr.sum(0).clamp(min=1.0)
        s._tvec=(Ttr.t()@h)/tdeg.unsqueeze(1)               # cache target embeddings for attribution
        ddeg=tgt_bin.sum(1).clamp(min=1.0)
        agg=(tgt_bin@s._tvec)/ddeg.unsqueeze(1)
        return s.norm(h+s.lin(agg))

class TargetDDI(nn.Module):
    # mode: "molecular" | "target_only" | "gated" | "graph" (gated + shared-target edge)
    def __init__(s,dmol,dtgt,C,cfg,mode="gated",add_target=None):
        super().__init__(); h=cfg.hidden
        if add_target is not None: mode="gated" if add_target else "molecular"
        s.mode=mode; s.use_tgt=(mode in ("gated","graph","target_only")) and dtgt>0
        if s.mode!="molecular" and not s.use_tgt: s.mode="molecular"
        s.mol=Enc(dmol,h,cfg.dropout)
        if s.use_tgt:
            s.tgt=Enc(dtgt,h,cfg.dropout)
            if s.mode in ("gated","graph"): s.gate=nn.Sequential(nn.Linear(2*h+1,h),nn.ReLU(),nn.Linear(h,h))
            if s.mode=="graph": s.stl=SharedTargetLayer(h)
        s.head=nn.Sequential(nn.Linear(3*h,cfg.pair_hidden),nn.BatchNorm1d(cfg.pair_hidden),
                             nn.ReLU(),nn.Dropout(cfg.dropout),nn.Linear(cfg.pair_hidden,C))
    def nodes(s,T):
        m=s.mol(T["mol"])
        if not s.use_tgt: return m
        t=s.tgt(T["tgt"])
        if s.mode=="target_only": z=t
        else:
            g=torch.sigmoid(s.gate(torch.cat([m,t,T["tcount"]],-1))); z=g*t+(1-g)*m
        if s.mode=="graph": z=s.stl(z,T["tgt_bin"],T["train_mask"])
        return z
    def forward(s,T,pu,pv):
        z=s.nodes(T); zi,zj=z[pu],z[pv]
        return s.head(torch.cat([zi+zj,(zi-zj).abs(),zi*zj],-1))
    @torch.no_grad()
    def attribute_pair(s,T,i,j,topk=5):
        """MECHANISM ATTRIBUTION (graph mode): rank the targets SHARED by drugs i,j by how much
        removing each changes the pair\'s predicted logits. Faithful by construction (ablation).
        Returns list of (target_index, effect). This is what we validate against DrugBank mechanisms."""
        if s.mode!="graph": return []
        dev=T["mol"].device
        def logit(tb):
            Tt=dict(T); Tt["tgt_bin"]=tb
            z=s.nodes(Tt); return s.head(torch.cat([z[i]+z[j],(z[i]-z[j]).abs(),z[i]*z[j]],-1))
        base=logit(T["tgt_bin"])
        shared=torch.where((T["tgt_bin"][i]*T["tgt_bin"][j])>0)[0].tolist()
        eff={}
        for t in shared:
            tb=T["tgt_bin"].clone(); tb[i,t]=0; tb[j,t]=0
            eff[t]=float((base-logit(tb)).abs().sum().item())
        return sorted(eff.items(),key=lambda kv:-kv[1])[:topk]

class AsymmetricLoss(nn.Module):
    def __init__(s,gn,gp,clip,eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logit,y):
        xs=torch.sigmoid(logit);xp=xs;xn=(1-xs)
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        l=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y);g=s.gp*y+s.gn*(1-y);return -(l*torch.pow(1-pt,g)).sum(1).mean()

from torch.utils.data import DataLoader,TensorDataset
def loader(part,id2n,C,bs,sh):
    pu=torch.tensor([id2n[u] for u in part["u"]]);pv=torch.tensor([id2n[v] for v in part["v"]])
    return DataLoader(TensorDataset(pu,pv,torch.tensor(multihot(part["y"],C))),batch_size=bs,shuffle=sh)
@torch.no_grad()
def predict(model,part,T,id2n,C):
    model.eval();z=model.nodes(T);out=[]
    for pu,pv,_ in loader(part,id2n,C,CFG.eval_batch,False):
        zi,zj=z[pu.to(DEVICE)],z[pv.to(DEVICE)]
        out.append(torch.sigmoid(model.head(torch.cat([zi+zj,(zi-zj).abs(),zi*zj],-1))).float().cpu().numpy())
    P=np.concatenate(out)
    return np.nan_to_num(P,nan=0.0,posinf=1.0,neginf=0.0)     # metrics never see NaN
def atomic_save(o,p): tmp=p+".tmp";torch.save(o,tmp);os.replace(tmp,p)

def train_one(model,sp,T,id2n,C,ckpt=None,log=False):
    opt=torch.optim.AdamW(model.parameters(),CFG.lr,weight_decay=CFG.weight_decay)
    warm=torch.optim.lr_scheduler.LambdaLR(opt,lambda e:min(1.0,(e+1)/max(1,CFG.warmup_epochs)))
    plateau=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=CFG.lr_factor,patience=CFG.lr_patience,min_lr=CFG.min_lr)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
    crit=AsymmetricLoss(CFG.asl_gamma_neg,CFG.asl_gamma_pos,CFG.asl_clip)
    dl=loader(sp["train"],id2n,C,CFG.batch_size,True);yv=multihot(sp["valid"]["y"],C)
    best=-1;best_ema=-1;bad=0;hist=[];start=0;best_state=None;ema=None
    if ckpt and os.path.exists(ckpt):
        try:
            st=torch.load(ckpt,map_location=DEVICE,weights_only=False)
            model.load_state_dict(st["model"]);opt.load_state_dict(st["opt"])
            warm.load_state_dict(st["warm"]);plateau.load_state_dict(st["plateau"]);scaler.load_state_dict(st["scaler"])
            start=st["epoch"]+1;best=st["best"];bad=st["bad"];hist=st["hist"];best_state=st["best_state"];ema=st.get("ema");best_ema=st.get("best_ema",best)
        except Exception as e: print("   ckpt unreadable, restart unit:",e)
    for ep in range(start,CFG.max_epochs):
        model.train();tl=0.0;t0=time.time()
        for pu,pv,y in dl:
            pu,pv,y=pu.to(DEVICE),pv.to(DEVICE),y.to(DEVICE);opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                logit=model(T,pu,pv)
            loss=crit(logit.float(),y.float())              # loss in fp32 (AMP-stable)
            if not torch.isfinite(loss):                     # skip a bad batch, never poison weights
                opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss).backward();scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),CFG.grad_clip);scaler.step(opt);scaler.update()
            tl+=loss.item()*len(pu)
        pv_=predict(model,sp["valid"],T,id2n,C);thr=tune_thresholds(yv,pv_);mv=evaluate(yv,pv_,thr)
        if ep<CFG.warmup_epochs: warm.step()
        else: plateau.step(mv["micro_auprc"])
        hist.append({"epoch":ep,"loss":tl/max(len(sp["train"]),1),**{f"val_{k}":v for k,v in mv.items()}})
        if log: print(f"    ep {ep:03d} loss {tl/max(len(sp['train']),1):.4f} val microAP {mv['micro_auprc']:.4f} "
                      f"macroF1 {mv['macro_f1']:.4f} lr {opt.param_groups[0]['lr']:.1e} bad {bad} ({time.time()-t0:.0f}s)")
        cur=mv["micro_auprc"]
        ema=cur if ema is None else (CFG.ema_beta*ema+(1-CFG.ema_beta)*cur)   # smoothed selection signal
        # keep the TRUE best raw-val checkpoint (that is what we evaluate on test),
        # but drive early-stopping off the SMOOTHED metric so noise can't trigger a false stop.
        if cur>best+CFG.min_delta:
            best=cur
            best_state={"model":{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},"thr":thr,"ema":ema}
        if ema>best_ema+CFG.min_delta:
            best_ema=ema;bad=0
        else: bad+=1
        if log: print(f"      (ema {ema:.4f} bestema {best_ema:.4f})")
        if ckpt: atomic_save({"model":model.state_dict(),"opt":opt.state_dict(),"warm":warm.state_dict(),
                    "plateau":plateau.state_dict(),"scaler":scaler.state_dict(),"epoch":ep,"best":best,
                    "bad":bad,"hist":hist,"best_state":best_state,"ema":ema,"best_ema":best_ema},ckpt)
        if ep+1>=CFG.min_epochs and bad>=CFG.patience: break   # only allow stopping after min_epochs
    if best_state is not None:
        model.load_state_dict({k:v.to(DEVICE) for k,v in best_state["model"].items()});thr=best_state["thr"]
    return model,thr,hist

## 7. Self-test (green light) + coverage probe + main run

In [9]:
def selftest():
    # target-driven synthetic cold-start world; target model must beat mol-only on S3
    from sklearn.metrics import average_precision_score
    rng=np.random.default_rng(0);Nd,Ntg,C=250,80,12
    tg=(rng.random((Nd,Ntg))<0.06).astype(np.float32);mo=rng.standard_normal((Nd,48)).astype(np.float32)
    W=rng.standard_normal((Ntg,C)).astype(np.float32)
    P=[(i,j) for i in range(Nd) for j in range(i+1,Nd) if rng.random()<0.05]
    def lab(i,j):
        u=np.clip(tg[i]+tg[j],0,1);p=1/(1+np.exp(-(u@W)));y=(p>0.7).astype(np.float32)
        if y.sum()==0:y[p.argmax()]=1
        return y
    Y={p:lab(*p) for p in P};perm=rng.permutation(Nd);tr_d=set(perm[:170])
    tr=[p for p in P if p[0] in tr_d and p[1] in tr_d];te=[p for p in P if p[0] not in tr_d and p[1] not in tr_d]
    id2n={i:i for i in range(Nd)}
    T={"mol":torch.tensor(mo).to(DEVICE),"tgt":torch.tensor(tg).to(DEVICE),
       "tcount":torch.tensor(np.log1p(tg.sum(1,keepdims=True)).astype(np.float32)).to(DEVICE)}
    import pandas as pd
    mk=lambda L: pd.DataFrame([{"u":i,"v":j,"y":list(np.where(Y[(i,j)]==1)[0])} for i,j in L])
    sp={"train":mk(tr[:int(.9*len(tr))]),"valid":mk(tr[int(.9*len(tr)):]),"test":mk(te)}
    saved=CFG.max_epochs;CFG.max_epochs=60;res={}
    for at in [True,False]:
        set_seed(0);m=TargetDDI(48,Ntg,C,CFG,add_target=at).to(DEVICE)
        m,thr,h=train_one(m,sp,T,id2n,C)
        p=predict(m,sp["test"],T,id2n,C);res[at]=evaluate(multihot(sp["test"]["y"],C),p,thr)["micro_auprc"]
    CFG.max_epochs=saved
    print(f"  self-test S3 micro-AP: WITH target={res[True]:.3f}  mol-only={res[False]:.3f}  Δ={res[True]-res[False]:+.3f}")
    assert res[True]>res[False]+0.05,"target branch not helping on rigged cold-start -> bug"
    assert res[False]>0.05,"mol-only degenerate -> bug"
    print("  SELF-TEST PASSED")
print("=== SELF-TEST ===");selftest()

=== SELF-TEST ===
  self-test S3 micro-AP: WITH target=0.802  mol-only=0.533  Δ=+0.269
  SELF-TEST PASSED


## 8. Run: probe coverage, then S2/S3-first grid (target vs no-target ablation)

### 3b. Extension slot — SIDER / `dex_ddi` (not in the uploaded notebook)

Paste the setup code for your SIDER features and `dex_ddi` baseline variant here. To
plug into the priority loop below with no further changes:
- Append any new molecular/target feature bank the same way the base cell does
  (`MOLBANK["your_key"] = ...` / `TGT_BANK["your_key"] = ...`).
- Append the variant tuple(s) to `VARIANTS` using the same
  `(name, mode, mol_key)` convention the base notebook uses, e.g.
  `VARIANTS.append(("graph_target_sider", "graph", "full_sider"))`.
- Add the variant name to `EXTRA_VARIANT_NAMES` in the next cell so the priority
  builder knows to schedule it (tier defaults to "S3 optional" / "S1 optional" —
  edit `TIER_OVERRIDES` there if it should be treated as core).


In [10]:
EXTRA_VARIANT_NAMES = []   # e.g. ["graph_target_sider", "dex_ddi"] once pasted above

## 8 (replaces original). Priority-ordered, missing-only run

Rebuilds the same feature banks and `VARIANTS` list your original Section 8 cell
built (unchanged logic, reused below), then runs `run_unit` only in this order:

1. **S3 core** — `full_target`, `shared_target_graph`, `ablation_no_target` × headline
   seeds (this is the ablation ladder the paper's claim depends on, on the hardest split)
2. **S3 optional** — `mol_only_classic`, `graph_esm`, `graph_both`, + anything in
   `EXTRA_VARIANT_NAMES` × ablation seeds
3. **S1 core** — same three core variants × headline seeds, on warm-start (shows the
   target branch doesn't hurt when nothing is cold-start)
4. **S1 optional** — same optional set as tier 2, on S1

Every `run_unit` call is already a no-op if `res_*.json` exists, so re-running this
cell after a session gets cut off is always safe and always resumes exactly where it
left off — including mid-training, via `train_one`'s own checkpoint load.

In [11]:
pairs,ds,C=load_dataset(CFG.dataset)
drug_ids=sorted(ds.keys(),key=str);id2n={d:i for i,d in enumerate(drug_ids)}
print(f"{CFG.dataset}: {len(pairs)} pairs | {len(drug_ids)} drugs | {C} classes")

# ---- TARGET DATA: categorized (target/enzyme/transporter/carrier) + coverage probe (hard gate) ----
print("\n=== TARGET COVERAGE PROBE ===")
cats = fetch_drug_targets_by_category(drug_ids) if CFG.enable_categorized_targets else {"target":fetch_drug_targets(drug_ids)}
d2p_union = {}
for cat,dd in cats.items():
    for d,ps in dd.items(): d2p_union.setdefault(d,set()).update(ps)
rep=coverage_report(drug_ids,d2p_union)
print(f"  coverage={rep['coverage']*100:.1f}%  covered={rep['covered']}/{rep['n_drugs']}  "
      f"median/drug={rep['median']:.0f}  distinct proteins={rep['distinct']}")
for cat,dd in cats.items():
    cr=coverage_report(drug_ids,dd)
    print(f"    [{cat}] coverage={cr['coverage']*100:.1f}%  distinct={cr['distinct']}")
TARGETS_OK = rep["coverage"]>=CFG.coverage_gate and rep["distinct"]>0
if not TARGETS_OK:
    print(f"  GATE FAILED: coverage < {CFG.coverage_gate*100:.0f}% or no targets fetched.")
else:
    print("  GATE PASSED -> target branch is viable.")

tgt_blocks=[]; tgt_names=[]
if TARGETS_OK:
    for cat in ["target","enzyme","transporter","carrier"]:
        Xc,keep=build_target_features(drug_ids,cats.get(cat,{}),CFG.target_min_freq,CFG.target_max)
        if Xc.shape[1]:
            tgt_blocks.append(Xc); tgt_names.append(f"{cat}({Xc.shape[1]})")
    print("  protein blocks: "+(", ".join(tgt_names) if tgt_names else "none"))

if TARGETS_OK and CFG.enable_go_pfam:
    all_prots=set().union(*d2p_union.values()) if d2p_union else set()
    prot2gopfam=fetch_go_pfam(all_prots)
    Xgo,Xpf,keep_go,keep_pf=build_go_pfam_features(drug_ids,d2p_union,prot2gopfam,CFG.go_pfam_min_freq,CFG.go_pfam_max)
    if Xgo.shape[1]: tgt_blocks.append(Xgo); tgt_names.append(f"GO({Xgo.shape[1]})")
    if Xpf.shape[1]: tgt_blocks.append(Xpf); tgt_names.append(f"Pfam({Xpf.shape[1]})")
    print(f"  +GO/Pfam blocks: GO={Xgo.shape[1]} Pfam={Xpf.shape[1]}")

Xtgt = np.concatenate(tgt_blocks,1).astype(np.float32) if tgt_blocks else np.zeros((len(drug_ids),0),np.float32)
print(f"  target-branch input width: {Xtgt.shape[1]}  [{', '.join(tgt_names) if tgt_names else 'none'}]")

Xmol_classic = molecular_features(drug_ids,ds)
mol_blocks=[Xmol_classic]; mol_names=[f"chemberta+morgan({Xmol_classic.shape[1]})"]

if CFG.enable_atc:
    print("\n=== ATC CLASSIFICATION ===")
    d2atc=fetch_atc(drug_ids)
    Xatc,keep_full,keep_l1=build_atc_features(drug_ids,d2atc,CFG.atc_min_freq)
    if Xatc.shape[1]:
        mol_blocks.append(Xatc); mol_names.append(f"ATC({Xatc.shape[1]}: {len(keep_full)} codes + {len(keep_l1)} L1 groups)")

if CFG.enable_physchem:
    sm=[ds[d] for d in drug_ids]
    Xpc=physchem_features(sm)
    mol_blocks.append(Xpc); mol_names.append(f"physchem({Xpc.shape[1]})")

if CFG.enable_cyp_roles:
    Xcyp,combo=fetch_cyp_roles(drug_ids)
    if Xcyp.shape[1]: mol_blocks.append(Xcyp); mol_names.append(f"CYP-roles({Xcyp.shape[1]})")

Xmol_full = np.concatenate(mol_blocks,1).astype(np.float32)
print(f"  molecular-branch input width: classic={Xmol_classic.shape[1]}  full={Xmol_full.shape[1]}  [{', '.join(mol_names)}]")

tcount = np.log1p(Xtgt.sum(1,keepdims=True)).astype(np.float32) if Xtgt.shape[1] else np.zeros((len(drug_ids),1),np.float32)
Xesm=np.zeros((len(drug_ids),0),np.float32); _esm_ok=False
MOLBANK={"classic":torch.tensor(Xmol_classic).to(DEVICE),"full":torch.tensor(Xmol_full).to(DEVICE)}
if TARGETS_OK and d2p_union:
    _prot_ids=sorted(set().union(*d2p_union.values()))
    _pidx={p:i for i,p in enumerate(_prot_ids)}
    _B=np.zeros((len(drug_ids),len(_prot_ids)),np.float32)
    for _i,_d in enumerate(drug_ids):
        for _p in d2p_union.get(_d,()): _B[_i,_pidx[_p]]=1.0
    TGT_BIN=torch.tensor(_B).to(DEVICE)
    SHARED=(_B@_B.T)
    if getattr(CFG,"use_esm",True):
        _esm=fetch_esm_embeddings(_prot_ids)
        Xesm,_esm_ok=build_esm_drug_features(drug_ids,d2p_union,_esm)
        print(f"   ESM-2 drug features: {Xesm.shape} (ok={_esm_ok})")
else:
    TGT_BIN=torch.zeros(len(drug_ids),0); SHARED=np.zeros((len(drug_ids),len(drug_ids)),np.float32)
TGT=torch.tensor(Xtgt).to(DEVICE); TCOUNT=torch.tensor(tcount).to(DEVICE)
_Xesm_t=torch.tensor(Xesm).to(DEVICE) if _esm_ok else torch.zeros(len(drug_ids),0).to(DEVICE)
TGT_BANK={"features":TGT,"esm":_Xesm_t,"both":(torch.cat([TGT,_Xesm_t],1) if _esm_ok else TGT)}
print(f"  features: molecular_classic={Xmol_classic.shape} molecular_full={Xmol_full.shape} target={Xtgt.shape}")

VARIANTS=[("full_target","gated","full"),
          ("shared_target_graph","graph","full"),
          ("ablation_no_target","molecular","full")]
if getattr(CFG,"run_target_only",False):
    VARIANTS.append(("target_only","target_only","full"))
if getattr(CFG,"run_esm_ablation",True) and _esm_ok:
    VARIANTS += [("graph_esm","graph","full"), ("graph_both","graph","full")]
if CFG.run_feature_ablation_ladder:
    VARIANTS.append(("mol_only_classic",False,"classic"))
# --- your pasted SIDER/dex_ddi variants (Section 3b) should already be appended to
# VARIANTS by the time this cell runs; nothing further needed here ---

def run_unit(split,vname,mode,mol_key,seed):
    tag=f"{CFG.dataset}_{CFG.feature_version}_{split}_{vname}_s{seed}";rp=f"{CFG.workdir}/res_{tag}.json"
    if os.path.exists(rp): return json.load(open(rp))
    sp,info=make_split(pairs,split,CFG.frac,seed)
    if len(sp["test"])==0 or len(sp["valid"])==0: print(f"  {split}: empty split"); return None
    audit(sp,split,id2n,None)
    tr_drugs=set(sp["train"]["u"])|set(sp["train"]["v"])
    tmask=torch.zeros(len(drug_ids),dtype=torch.bool)
    for d in tr_drugs: tmask[id2n[d]]=True
    _tbank={"graph_esm":"esm","graph_both":"both"}.get(vname,"features")
    _tgt=TGT_BANK.get(_tbank,TGT)
    T={"mol":MOLBANK[mol_key],"tgt":_tgt,"tcount":TCOUNT,"tgt_bin":TGT_BIN,"train_mask":tmask.to(DEVICE)}
    set_seed(seed);m=TargetDDI(MOLBANK[mol_key].shape[1],_tgt.shape[1],C,CFG,mode=mode).to(DEVICE)
    m,thr,hist=train_one(m,sp,T,id2n,C,ckpt=f"{CFG.workdir}/ck_{tag}.pt",log=True)
    pd.DataFrame(hist).to_csv(f"{CFG.workdir}/hist_{tag}.csv",index=False)
    yte=multihot(sp["test"]["y"],C);pt=predict(m,sp["test"],T,id2n,C)
    shr=np.array([SHARED[id2n[u],id2n[v]] for u,v in zip(sp["test"]["u"],sp["test"]["v"])],np.int32)
    res=evaluate(yte,pt,thr);np.savez_compressed(f"{CFG.workdir}/pred_{tag}.npz",y=yte,p=pt,shared=shr)
    res.update(dict(dataset=CFG.dataset,split=split,variant=vname,seed=seed,mol_key=mol_key,mode=mode,
                    coverage=rep["coverage"],n_test=len(sp["test"])))
    json.dump(res,open(rp,"w"),indent=2)
    print(f"  [{tag}] microAP={res['micro_auprc']:.4f} macroF1={res['macro_f1']:.4f} PR-lift={res['pr_lift']:.1f}x")
    del m;gc.collect();torch.cuda.empty_cache();return res

# ---------------- priority-ordered, missing-only unit list ----------------
CORE_NAMES = {"full_target", "shared_target_graph", "ablation_no_target"}
OPTIONAL_NAMES = {v[0] for v in VARIANTS if v[0] not in CORE_NAMES} | set(EXTRA_VARIANT_NAMES)
VARIANT_LOOKUP = {v[0]: v for v in VARIANTS}

GRID_TIME_BUDGET_HOURS = None   # e.g. 8.0 to stop scheduling new units after 8h; None = no limit
_grid_start = time.time()

def _pending_units(split, names, seeds_fn):
    out = []
    for name in names:
        if name not in VARIANT_LOOKUP:
            print(f"  [skip] variant '{name}' not defined in VARIANTS -- paste its setup in Section 3b")
            continue
        _, mode, mol_key = VARIANT_LOOKUP[name]
        for seed in seeds_fn(name):
            tag = f"{CFG.dataset}_{CFG.feature_version}_{split}_{name}_s{seed}"
            rp = f"{CFG.workdir}/res_{tag}.json"
            out.append((split, name, mode, mol_key, seed, os.path.exists(rp)))
    return out

def _headline(name): return CFG.headline_seeds
def _ablation(name): return CFG.ablation_seeds

TIERS = [
    ("S3 core",     _pending_units("S3", sorted(CORE_NAMES), _headline)),
    ("S3 optional", _pending_units("S3", sorted(OPTIONAL_NAMES), _ablation)),
    ("S1 core",     _pending_units("S1", sorted(CORE_NAMES), _headline)),
    ("S1 optional", _pending_units("S1", sorted(OPTIONAL_NAMES), _ablation)),
]

print("\n" + "=" * 70)
for tier_name, units in TIERS:
    done = sum(1 for u in units if u[5]); todo = len(units) - done
    print(f"[{tier_name}] {done} already complete, {todo} pending")
print("=" * 70)

for tier_name, units in TIERS:
    print(f"\n--- TIER: {tier_name} ---")
    for split, vname, mode, mol_key, seed, already_done in units:
        if already_done:
            continue    # run_unit would also just short-circuit on this, skip the log noise
        if GRID_TIME_BUDGET_HOURS is not None and (time.time() - _grid_start) / 3600 >= GRID_TIME_BUDGET_HOURS:
            print(f"  time budget ({GRID_TIME_BUDGET_HOURS}h) reached -- stopping before starting "
                  f"{split}/{vname}/s{seed}. Safe to stop here; re-run this cell next session to resume.")
            break
        try:
            run_unit(split, vname, mode, mol_key, seed)
        except Exception as e:
            import traceback; traceback.print_exc(); print(f"  FAILED {split}/{vname}/s{seed}: {e}")
    else:
        continue
    break   # time budget hit inside the inner loop -> stop scheduling further tiers too

print("\nMissing-only grid pass complete (or time budget reached).")


Downloading...
100%|██████████| 44.4M/44.4M [00:01<00:00, 42.9MiB/s]
Loading...
Done!


DrugBank: 191402 pairs | 1706 drugs | 86 classes

=== TARGET COVERAGE PROBE ===
   DrugBank categorized exports unavailable (likely needs an account) -> falling back to UniProt idmapping + EC/GO-based enzyme/transporter/target reconstruction (carrier has no comparable public signature, so that channel stays empty)
   UniProt idmapping: 1604 drugs with >=1 target (uncategorized)
   fetching EC/GO for 2047 proteins (enzyme/transporter reconstruction)...
   EC/GO: 20/2047 proteins (0s elapsed, 0 chunk failures so far)
   EC/GO: 120/2047 proteins (3s elapsed, 0 chunk failures so far)
   EC/GO: 220/2047 proteins (5s elapsed, 0 chunk failures so far)
   EC/GO: 320/2047 proteins (8s elapsed, 0 chunk failures so far)
   EC/GO: 420/2047 proteins (10s elapsed, 0 chunk failures so far)
   EC/GO: 520/2047 proteins (13s elapsed, 0 chunk failures so far)
   EC/GO: 620/2047 proteins (15s elapsed, 0 chunk failures so far)
   EC/GO: 720/2047 proteins (18s elapsed, 0 chunk failures so far)
   EC/GO: 820

## 9. Results + paired-bootstrap CI (does the target branch help on cold-start?)

In [12]:
rows=[json.load(open(p)) for p in sorted(glob.glob(f"{CFG.workdir}/res_*.json"))]
if not rows: print("no results")
else:
    R=pd.DataFrame(rows)
    agg=R.groupby(["split","variant"])[["micro_auprc","auprc_per_type","macro_f1","auroc_per_type","pr_lift","top1","top5"]].agg(["mean","std","count"]).reset_index()
    agg.columns=["_".join([c for c in col if c]).rstrip("_") for col in agg.columns]
    agg.to_csv(f"{CFG.workdir}/all_results.csv",index=False);R.to_csv(f"{CFG.workdir}/all_runs_raw.csv",index=False)
    print("Saved all_results.csv\n")
    for split in [s for s in CFG.splits if s in set(R.split)]:
        print(f"--- {split} ---")
        for v in R[R.split==split].variant.unique():
            q=agg[(agg.split==split)&(agg.variant==v)]
            if len(q):
                m=q["micro_auprc_mean"].iloc[0];s=q["micro_auprc_std"].iloc[0] if q["micro_auprc_count"].iloc[0]>1 else float("nan")
                print(f"  {v:20s} microAP={m:.4f}"+(f"±{s:.4f}" if s==s else "")+f"  macroF1={q['macro_f1_mean'].iloc[0]:.4f}")
        # bootstrap CI: full_target vs no_target, seed 0
        ff=f"{CFG.workdir}/pred_{CFG.dataset}_{CFG.feature_version}_{split}_full_target_s0.npz"
        bb=f"{CFG.workdir}/pred_{CFG.dataset}_{CFG.feature_version}_{split}_ablation_no_target_s0.npz"
        if os.path.exists(ff) and os.path.exists(bb):
            try:
                print(f"  computing paired-bootstrap CI for {split} (this can take a while, watch for [bootstrap] progress lines below)...")
                F_=np.load(ff);B_=np.load(bb)
                pt,lo,hi=paired_bootstrap(F_["y"],F_["p"],B_["p"],CFG.n_bootstrap)
                verdict="TARGETS HELP (CI>0)" if lo>0 else ("TARGETS HURT (CI<0)" if hi<0 else "INCONCLUSIVE (CI spans 0)")
                print(f"  Δ microAP (target − no-target) = {pt:+.4f}  95% CI [{lo:+.4f},{hi:+.4f}]  -> {verdict}")
            except Exception as e:
                print("  (bootstrap skipped:",e,")")
        print()

print("\n"+"="*60+"\nRUN COMPLETE — results in "+CFG.workdir+"/all_results.csv\n"+"="*60)

# ================= TARGET-OVERLAP STRATIFICATION (the mechanism figure) =================
# On S3 (both drugs unseen), does the shared-target GRAPH win *more* as the pair shares
# more targets? That is the causal story: performance tracks shared-target connectivity.
def _micro_ap(y,p):
    from sklearn.metrics import average_precision_score
    y=y.ravel(); p=p.ravel()
    return float(average_precision_score(y,p)) if y.sum()>0 else float("nan")
def stratify(split="S3"):
    import glob as _g
    fg=sorted(_g.glob(f"{CFG.workdir}/pred_{CFG.dataset}_{CFG.feature_version}_{split}_shared_target_graph_s0.npz"))
    fb=sorted(_g.glob(f"{CFG.workdir}/pred_{CFG.dataset}_{CFG.feature_version}_{split}_ablation_no_target_s0.npz"))
    if not fg or not fb: print(f"  [stratify {split}] need graph + no_target seed-0 preds"); return
    G=np.load(fg[0]); B=np.load(fb[0])
    if "shared" not in G.files: print("  [stratify] no shared-count saved (re-run needed)"); return
    shr=G["shared"]
    buckets=[("0",shr==0),("1",shr==1),("2-3",(shr>=2)&(shr<=3)),("4+",shr>=4)]
    print(f"\n  target-overlap stratification on {split} (micro-AP):")
    print(f"    {'shared':8s}{'n':>8s}{'no_target':>12s}{'graph':>10s}{'Δ':>9s}")
    for name,mask in buckets:
        if mask.sum()==0: continue
        ap_b=_micro_ap(B["y"][mask],B["p"][mask]); ap_g=_micro_ap(G["y"][mask],G["p"][mask])
        print(f"    {name:8s}{int(mask.sum()):>8d}{ap_b:>12.4f}{ap_g:>10.4f}{ap_g-ap_b:>+9.4f}")
for _sp in [x for x in ("S2","S3") if x in getattr(CFG,"splits",[])]:
    try: stratify(_sp)
    except Exception as e: print("  stratify error:",e)


Saved all_results.csv

--- S2 ---
  ablation_no_target   microAP=0.6797±0.0197  macroF1=0.4802
  dex_ddi              microAP=0.7189±0.0056  macroF1=0.5320
  full_target          microAP=0.7620±0.0145  macroF1=0.5758
  graph_both           microAP=0.7582  macroF1=0.5873
  graph_esm            microAP=0.7528  macroF1=0.5729
  graph_target_sider   microAP=0.7794  macroF1=0.5976
  mol_only_classic     microAP=0.6622  macroF1=0.4944
  shared_target_graph  microAP=0.7566±0.0098  macroF1=0.5863
  computing paired-bootstrap CI for S2 (this can take a while, watch for [bootstrap] progress lines below)...
    [bootstrap] N=53911 pairs x C=86 classes, nb=2000 resamples -- timing first 10 iters...
    [bootstrap] ~2007ms/iter -> est. total 66.9 min for nb=2000
    [bootstrap] 100/2000 (5%) elapsed=201s  eta=3826s
    [bootstrap] 200/2000 (10%) elapsed=402s  eta=3620s
    [bootstrap] 300/2000 (15%) elapsed=604s  eta=3420s
    [bootstrap] 400/2000 (20%) elapsed=806s  eta=3223s
    [bootstrap] 500/2

## 10. What this answers

The run prints, for S2 and S3 (cold-start), a paired-bootstrap CI on the gap between the
target-augmented model and the identical model without targets. That single line is the
experiment: **if the CI is above zero on S2/S3, protein targets are the missing input and
you have a real, mechanistically-motivated contribution.** If it spans zero, targets are
not the lever for this dataset, and you have learned that cleanly rather than after days
of blind architecture search.

Order matters here: S2/S3 run first, so even if the session is time-limited you get the
cold-start answer before S1 (which we already know saturates ~0.29 regardless of model).

Outputs in `/kaggle/working`: `all_results.csv`, `all_runs_raw.csv`, per-unit JSON,
`hist_*.csv`, `pred_*.npz`, checkpoints. Resumable: re-run to skip finished units.

If targets help here, phase two is the CID->DrugBank bridge to carry the same feature to
TWOSIDES. If they do not, the honest paper is the leakage-clean S1/S2/S3 benchmark plus
the finding that neither architecture nor target features beat strong featurized baselines
on cold-start — itself a real contribution.

Send me `all_results.csv` and the CI lines and I will write the Results around the real numbers.

### Fix applied: checkpoint path/key

The original cell loads `ck_..._s{seed}.pt.best`, which `train_one` never writes —
only `ck_{tag}.pt` exists, and the best weights live inside it at
`checkpoint["best_state"]["model"]` (the top-level `"model"` key is the *last* epoch's
live weights, not necessarily the best one selected by early stopping). Fixed below;
everything else is the original attribution-validation logic, unchanged.

In [13]:
import re as _re

CYP_UNIPROT = {
 "CYP1A2":"P05177","CYP2B6":"P20813","CYP2C8":"P10632","CYP2C9":"P11712",
 "CYP2C19":"P33261","CYP2D6":"P10635","CYP2E1":"P05181","CYP3A4":"P08684","CYP3A5":"P20815",
 "CYP2A6":"P11509","CYP2J2":"P51589","CYP1A1":"P04798","CYP1B1":"Q16678",
}
def mechanism_targets_from_text(txt):
    if not isinstance(txt,str): return set()
    acc=set()
    for gene in _re.findall(r"CYP\s?\d+[A-Z]\d+", txt.upper().replace(" ","")):
        g=gene.replace(" ","")
        if g in CYP_UNIPROT: acc.add(CYP_UNIPROT[g])
    return acc

def load_best_model(tag, dmol, dtgt, mode="graph"):
    '''FIX: read ck_{tag}.pt (the only file train_one ever writes) and load the
    BEST state dict cached at checkpoint["best_state"]["model"] -- not the top-level
    "model" key, which is just whatever epoch was running when training stopped.'''
    ck_path = f"{CFG.workdir}/ck_{tag}.pt"
    if not os.path.exists(ck_path):
        return None
    state = torch.load(ck_path, map_location=DEVICE, weights_only=False)
    best_state = state.get("best_state")
    if best_state is None or "model" not in best_state:
        print(f"  [attribution] {tag}: checkpoint has no best_state yet (still early in training?)")
        return None
    model = TargetDDI(dmol, dtgt, C, CFG, mode=mode).to(DEVICE)
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state["model"].items()})
    model.eval()
    return model

def run_attribution_validation(split="S3", seed=0, max_pairs=400):
    tag = f"{CFG.dataset}_{CFG.feature_version}_{split}_shared_target_graph_s{seed}"
    model = load_best_model(tag, MOLBANK["full"].shape[1], Xtgt.shape[1], mode="graph")
    if model is None:
        print(f"  [attribution] no usable checkpoint for {split} seed{seed}; run the grid first")
        return
    try:
        raw=DDI(name=CFG.dataset).get_data()
        desc_col=[c for c in raw.columns if raw[c].dtype==object and raw[c].astype(str).str.contains("CYP",case=False,na=False).any()]
    except Exception as e:
        print("  [attribution] could not load DDI descriptions:",e); return
    if not desc_col:
        print("  [attribution] no mechanism/description text with CYP mentions in this dataset build; skipping")
        return
    dc=desc_col[0]; cols={c.lower():c for c in raw.columns}
    id1=cols.get("drug1_id","Drug1_ID"); id2=cols.get("drug2_id","Drug2_ID")
    pair_mech={}
    for a,b,t in zip(raw[id1],raw[id2],raw[dc]):
        m=mechanism_targets_from_text(t)
        if m: pair_mech[(str(a),str(b))]=m; pair_mech[(str(b),str(a))]=m

    sp,_=make_split(pairs,split,CFG.frac,seed)
    tr_drugs=set(sp["train"]["u"])|set(sp["train"]["v"])
    tmask=torch.zeros(len(drug_ids),dtype=torch.bool)
    for d in tr_drugs: tmask[id2n[d]]=True
    T={"mol":MOLBANK["full"],"tgt":TGT,"tcount":TCOUNT,"tgt_bin":TGT_BIN,"train_mask":tmask.to(DEVICE)}

    prot_ids=_prot_ids
    hits=tot=0
    for u,v in zip(sp["test"]["u"],sp["test"]["v"]):
        mech=pair_mech.get((str(u),str(v)))
        if not mech: continue
        i,j=id2n[u],id2n[v]
        ranked=model.attribute_pair(T,i,j,topk=3)
        if not ranked: continue
        top_prots={prot_ids[t] for t,_ in ranked}
        tot+=1; hits+= 1 if (top_prots & mech) else 0
        if tot>=max_pairs: break
    if tot==0:
        print(f"  [attribution/{split}] no test pairs with a parseable CYP mechanism AND shared targets"); return
    print(f"  [MECHANISM ATTRIBUTION / {split}] top-3 attribution matches known CYP mechanism "
          f"in {hits}/{tot} = {hits/tot:.3f} of pairs")

for _sp in [x for x in ("S2","S3") if x in getattr(CFG,"splits",[])]:
    try: run_attribution_validation(_sp)
    except Exception as e:
        import traceback; traceback.print_exc(); print("  attribution error:",e)


Found local copy...
Loading...
Done!
Found local copy...
Loading...


  [attribution] no mechanism/description text with CYP mentions in this dataset build; skipping


Done!


  [attribution] no mechanism/description text with CYP mentions in this dataset build; skipping


## 11. Snapshot (new — did not exist before)

Zips the small, essential artifacts (`res_*.json`, `all_results.csv`,
`all_runs_raw.csv`, `hist_*.csv`) separately from the large checkpoint files
(`ck_*.pt`, often hundreds of MB each), so you can always grab the results bundle
quickly and only pull checkpoints if you actually need to resume or rerun attribution
later. Run this before a session ends, whether or not the grid finished — a partial
results bundle is still worth keeping.

In [14]:
import zipfile, glob, time as _t

stamp = _t.strftime("%Y%m%d_%H%M%S")
results_zip = f"{CFG.workdir}/results_bundle_{stamp}.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ("res_*.json", "all_results.csv", "all_runs_raw.csv", "hist_*.csv"):
        for f in glob.glob(f"{CFG.workdir}/{pattern}"):
            z.write(f, arcname=os.path.basename(f))
print(f"results bundle: {results_zip}  ({os.path.getsize(results_zip)/1e6:.1f} MB)")

SAVE_CHECKPOINTS_TOO = False   # set True if you want ck_*.pt bundled as well (can be large)
if SAVE_CHECKPOINTS_TOO:
    ckpt_zip = f"{CFG.workdir}/checkpoints_bundle_{stamp}.zip"
    with zipfile.ZipFile(ckpt_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(f"{CFG.workdir}/ck_*.pt"):
            z.write(f, arcname=os.path.basename(f))
    print(f"checkpoint bundle: {ckpt_zip}  ({os.path.getsize(ckpt_zip)/1e6:.1f} MB)")

print("\nDownload via the Kaggle Output pane, or commit this session's output as a "
      "Kaggle dataset to carry results/checkpoints into your next session cleanly.")


results bundle: /root/ddi_work/results_bundle_20260918_054018.zip  (0.6 MB)

Download via the Kaggle Output pane, or commit this session's output as a Kaggle dataset to carry results/checkpoints into your next session cleanly.


In [ ]:
# ===== PERSIST OUTPUTS to /kaggle/working (survives Kaggle commit) =====
import os, glob, shutil
_WD = CFG.workdir if "CFG" in dir() else os.path.expanduser("~/ddi_work")
_DEST = "/kaggle/working/ddi_out" if os.path.isdir("/kaggle/working") else os.path.abspath("./ddi_out")
os.makedirs(_DEST, exist_ok=True); _n=0
for pat in ["all_results.csv","all_runs_raw.csv","res_*.json","pred_*.npz","hist_*.csv","results_bundle_*.zip"]:
    for f in glob.glob(os.path.join(_WD,pat)):
        try: shutil.copy2(f,_DEST); _n+=1
        except Exception as e: print("copy failed",f,e)
print(f"persisted {_n} files -> {_DEST}")
print("all_results.csv present:", os.path.exists(os.path.join(_DEST,"all_results.csv")))
